# 08b - Single Reid Network Auxetic Optimization with MetaForge LAMMPS

Focus on one Reid network and search stiffness interventions that make its real MetaForge p-ratio lower, ideally negative. The candidate generator is configurable so the same code can later be expanded to multiple networks.


In [ ]:
# Imports.

import copy
import json
import os
import random
import shutil
import sys
from pathlib import Path

os.environ.setdefault('MPLCONFIGDIR', '/tmp/matplotlib')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

META_FORGE_SRC = Path('/home/alexz/MetaForge/src')
if str(META_FORGE_SRC) not in sys.path:
    sys.path.insert(0, str(META_FORGE_SRC))

from graph_utils import calc_p_ratio_rollout_sides, directional_side_indices_from_box, draw_graph, remove_periodic_edges
from lss.plotting import PAPER_COLORS, apply_editorial_style, style_axes

from auxetic import ElasticRunConfig, FastElasticSettings, run_elastic_simulation

apply_editorial_style()

try:
    display
except NameError:
    def display(*args, **kwargs):
        return None



In [ ]:
# Config.

seed = 8021
random.seed(seed)
rng = np.random.default_rng(seed)

ROOT = Path.cwd()
if not (ROOT / 'data').exists():
    ROOT = ROOT.parent

cfg = {
    'dataset_path': str(ROOT / 'data' / 'new_reid_combined.pt'),
    'source_08_descriptors': str(ROOT / 'notebooks' / 'results' / 'reid_stiffness_pratio_affine_rollout' / 'reid_static_descriptors.csv'),
    'output_dir': str(ROOT / 'notebooks' / 'results' / 'reid_stiffness_pratio_optimization_lammps'),
    # None means: choose the highest-p-ratio Reid network as the starting point.
    'single_sim_idx': None,
    'select_by': 'final_p_ratio',
    'target_frame': 100,
    'side_quantile': 0.10,
    'optimization_feature': 'stiffness_std',
    'static_prediction_features': ['stiffness_mean', 'stiffness_std'],
    # Sample one ordered edge list; diagonal edges are 4x more likely to appear early.
    'random_candidate_count': 12,
    'diagonal_sample_weight': 4.0,
    # Scale the whole network after softening the first k sampled edges.
    'support_factor_grid': [1.25, 1.5, 2.0, 3.0, 4.0],
    'support_target': 'all',
    'near_zero_stiffness': 1e-7,
    'min_stiffness': 1e-7,
    'max_stiffness': 2.0,
    'lammps_cmd': 'lmp',
    'run_lammps': True,
    # Fast screening uses looser minimization and faster finite-strain deformation.
    # Keep this True for MCMC; final validation below temporarily disables it.
    'lammps_fast_screening': True,
    'final_lammps_fast_screening': False,
    'lammps_fast_maxiter': 1000,
    'lammps_fast_maxeval': 4000,
    'lammps_fast_ftol': 1e-8,
    'lammps_fast_dmax': 5e-2,
    'lammps_fast_srate': 1e-3,
    'lammps_screen': 'none',
    # The MCMC changes only spring constants, not topology/rest lengths.
    # Reuse one network.lmp and write only bond_coeffs.mod for each LAMMPS call.
    'lammps_reuse_network_topology': True,
    'valid_bulk_min': 1e-4,
    'valid_shear_min': 1e-4,
    # Stronger physical screen: reject candidates that only look auxetic because
    # bulk/shear collapsed relative to the original network.
    'valid_min_bulk_retention': 0.01,
    'valid_min_shear_retention': 0.02,
    'mass': 1e6,
    'angles': 0.0,
}

output_dir = Path(cfg['output_dir'])
run_dir = output_dir / 'lammps_runs_single_network'
output_dir.mkdir(parents=True, exist_ok=True)
run_dir.mkdir(parents=True, exist_ok=True)
(output_dir / 'config.json').write_text(json.dumps(cfg, indent=2))
print(json.dumps(cfg, indent=2))


In [ ]:
# Reversible response-biased MCMC for stiffness optimization.
#
# Main idea:
#   - Propose random single-edge stiffness changes.
#   - Run MetaForge/LAMMPS for every proposal.
#   - Accept/reject by the true LAMMPS elastic objective:
#       score = log(G/B) - mechanical penalties.
#   - Lower p-ratio means larger G/B, so this targets lower p-ratio directly.

cfg.update({
    # MCMC.
    'rev_mcmc_steps': 2000,
    'rev_mcmc_burn_in': 100,
    'rev_mcmc_keep_every': 10,
    'rev_mcmc_temperature': 0.03,

    # LAMMPS-in-the-loop mode. Random edge proposals are scored by true MetaForge G/B.
    'rev_lammps_mcmc_steps': 1000,
    'rev_lammps_burn_in': 0,
    'rev_lammps_keep_every': 10,
    'rev_lammps_temperature': 0.005,
    'rev_lammps_min_shear_retention': 0.55,
    'rev_lammps_shear_retention_penalty': 4.0,
    'rev_lammps_invalid_penalty': 100.0,

    # Random move probabilities for LAMMPS-scored proposals.
    'rev_prob_weaken': 0.55,
    'rev_prob_strengthen': 0.45,
    'rev_prob_restore': 0.0,

    # Bolder move sizes. Small 0.9x/1.1x moves left G/B almost unchanged.
    'rev_weaken_factor': 0.25,
    'rev_strengthen_factor': 2.0,
    'rev_restore_mix': 0.75,

    # Often prune close to zero instead of just weakening.
    'rev_prob_prune_in_weaken_move': 0.25,
    'rev_near_zero_value': cfg.get('near_zero_stiffness', 1e-7),

    # Candidate batch size for response-biased proposals.
    'rev_candidate_count': 32,

    # Proposal bias from approximate response.
    # Weaken/prune score: high ΔB, low ΔG.
    'rev_weaken_deltaB_weight': 1.0,
    'rev_weaken_deltaG_penalty': 8.0,
    'rev_weaken_softmax_temperature': 0.05,

    # Strengthen score: edges important for shear, or edges currently too weak.
    'rev_strengthen_deltaG_weight': 1.0,
    'rev_strengthen_deltaB_penalty': 2.0,
    'rev_strengthen_weak_edge_bonus': 0.25,
    'rev_strengthen_softmax_temperature': 0.05,

    # Paired response move: soften bulk-sensitive edge + stiffen shear-efficient edge.
    'rev_pair_candidate_count': 24,
    'rev_pair_ratio_bonus': 2.0,
    'rev_pair_softmax_temperature': 0.05,

    # Linear response.
    'linear_response_strain': 1e-4,
    'linear_response_rcond': 1e-10,
    'linear_response_boundary_quantile': 0.10,

    # Target distribution learned from low-p-ratio networks.
    'rev_target_p_ratio_cutoff': -0.10,
    'rev_min_target_count': 5,
    'rev_target_quantile_fallback': 0.20,
    'rev_target_feature_cols': ['linear_log_g_over_b'],
    'rev_target_distribution_weight': 0.25,

    # Objective. The Hessian/linear response is useful, but only if shear is
    # not destroyed while bulk drops.
    'rev_log_g_over_b_weight': 1.0,
    'rev_min_shear_retention': 0.85,
    'rev_shear_retention_penalty': 25.0,

    # Physical penalties.
    'mcmc_active_threshold': 1e-5,
    'rev_max_near_zero_frac': 0.45,
    'rev_near_zero_penalty': 6.0,
    'rev_low_degree_min': 2,
    'rev_low_degree_penalty': 0.5,
    'rev_log_stiffness_drift_penalty': 0.03,

    # Hard safety filters.
    'rev_require_active_connected': False,
    'rev_min_active_degree_after_prune': 2,

    # Final LAMMPS validation. The Hessian MCMC generates many cheap
    # candidates; LAMMPS is used on a small diverse budget and cached.
    'rev_lammps_budget': 12,
    'rev_lammps_prefilter_count': 60,
    'rev_lammps_diversity_min_logdist': 0.02,
    'rev_top_k_to_simulate': 12,

    # None means start from highest-p-ratio network.
    'rev_start_sim_idx': cfg.get('single_sim_idx', None),
})


def unique_edge_stiffness(graph):
    edge_index = graph.edge_index.detach().cpu().numpy()
    edge_attr = graph.edge_attr.detach().cpu().numpy()

    pair_to_stiffness = {}

    for e in range(edge_index.shape[1]):
        i, j = int(edge_index[0, e]), int(edge_index[1, e])
        key = tuple(sorted((i, j)))

        if key not in pair_to_stiffness:
            pair_to_stiffness[key] = float(edge_attr[e, -1])

    keys = list(pair_to_stiffness.keys())
    values = np.asarray([pair_to_stiffness[k] for k in keys], dtype=float)

    return keys, values


def update_graph_stiffness(graph, keys, new_values):
    out = copy.deepcopy(graph)

    value_by_key = {
        key: float(value)
        for key, value in zip(keys, new_values)
    }

    edge_index = out.edge_index.detach().cpu().numpy()
    edge_attr = out.edge_attr.clone().detach().cpu()

    for e in range(edge_index.shape[1]):
        i, j = int(edge_index[0, e]), int(edge_index[1, e])
        key = tuple(sorted((i, j)))
        edge_attr[e, -1] = value_by_key[key]

    out.edge_attr = edge_attr

    return out


def unique_edge_geometry(graph, keys):
    xy = graph.x[:, :2].detach().cpu().numpy()

    x1, x2 = float(graph.box.x1), float(graph.box.x2)
    y1, y2 = float(graph.box.y1), float(graph.box.y2)

    cx, cy = 0.5 * (x1 + x2), 0.5 * (y1 + y2)
    sx, sy = max(abs(x2 - x1), 1e-12), max(abs(y2 - y1), 1e-12)

    rows = []

    for edge_idx, (i, j) in enumerate(keys):
        pi, pj = xy[int(i)], xy[int(j)]

        midpoint = 0.5 * (pi + pj)
        vec = pj - pi

        angle = abs(np.arctan2(vec[1], vec[0]))
        angle = min(angle, np.pi - angle)

        if angle < np.deg2rad(25):
            orientation = 'horizontal'
        elif angle > np.deg2rad(65):
            orientation = 'vertical'
        else:
            orientation = 'diagonal'

        x_norm = (midpoint[0] - cx) / (0.5 * sx)
        y_norm = (midpoint[1] - cy) / (0.5 * sy)
        r_norm = float(np.sqrt(x_norm * x_norm + y_norm * y_norm))

        if r_norm < 0.35:
            region = 'center'
        elif r_norm > 0.75:
            region = 'boundary'
        else:
            region = 'middle'

        rows.append({
            'edge_idx': int(edge_idx),
            'i': int(i),
            'j': int(j),
            'orientation': orientation,
            'region': region,
            'r_norm': r_norm,
            'angle_deg': float(np.rad2deg(angle)),
            'mid_x_norm': float(x_norm),
            'mid_y_norm': float(y_norm),
        })

    return pd.DataFrame(rows)


def graph_area_from_box(graph):
    x1, x2 = float(graph.box.x1), float(graph.box.x2)
    y1, y2 = float(graph.box.y1), float(graph.box.y2)

    return max(abs(x2 - x1) * abs(y2 - y1), 1e-12)


def boundary_node_mask_from_box(graph, quantile=None):
    if quantile is None:
        quantile = float(cfg['linear_response_boundary_quantile'])

    xy = graph.x[:, :2].detach().cpu().numpy()

    x = xy[:, 0]
    y = xy[:, 1]

    x_lo = np.quantile(x, quantile)
    x_hi = np.quantile(x, 1.0 - quantile)
    y_lo = np.quantile(y, quantile)
    y_hi = np.quantile(y, 1.0 - quantile)

    return (
        (x <= x_lo)
        | (x >= x_hi)
        | (y <= y_lo)
        | (y >= y_hi)
    )


def relaxed_spring_energy_linear_response_constrained(
    xy,
    keys,
    values,
    strain_matrix,
    boundary_mask,
):
    xy = np.asarray(xy, dtype=float)
    values = np.asarray(values, dtype=float)
    boundary_mask = np.asarray(boundary_mask, dtype=bool)

    n_nodes = xy.shape[0]
    threshold = float(cfg['mcmc_active_threshold'])

    free_nodes = np.where(~boundary_mask)[0]
    free_dof_by_node = {}

    col = 0
    for node in free_nodes:
        free_dof_by_node[int(node)] = (col, col + 2)
        col += 2

    n_free_dof = col

    rows = []
    rhs = []

    for edge_idx, (i, j) in enumerate(keys):
        k = float(values[edge_idx])

        if k <= threshold:
            continue

        i, j = int(i), int(j)

        r = xy[j] - xy[i]
        length = float(np.linalg.norm(r))

        if length <= 1e-12:
            continue

        n = r / length
        sqrt_k = np.sqrt(k)

        row = np.zeros(n_free_dof, dtype=float)

        if i in free_dof_by_node:
            c0, c1 = free_dof_by_node[i]
            row[c0:c1] -= sqrt_k * n

        if j in free_dof_by_node:
            c0, c1 = free_dof_by_node[j]
            row[c0:c1] += sqrt_k * n

        affine_extension = sqrt_k * float(n @ (strain_matrix @ r))

        rows.append(row)
        rhs.append(affine_extension)

    if not rows:
        return 0.0

    A = np.vstack(rows)
    b = np.asarray(rhs, dtype=float)

    if A.shape[1] == 0:
        residual = b
    else:
        u, *_ = np.linalg.lstsq(
            A,
            -b,
            rcond=float(cfg['linear_response_rcond']),
        )
        residual = A @ u + b

    energy = 0.5 * float(residual @ residual)

    return energy


def linear_response_moduli_for_values(graph, keys, values):
    xy = graph.x[:, :2].detach().cpu().numpy()
    area = graph_area_from_box(graph)
    eps = float(cfg['linear_response_strain'])

    boundary_mask = boundary_node_mask_from_box(
        graph,
        quantile=float(cfg['linear_response_boundary_quantile']),
    )

    E_bulk = eps * np.eye(2)

    E_pure_shear = np.array([
        [-eps, 0.0],
        [0.0, eps],
    ], dtype=float)

    bulk_energy = relaxed_spring_energy_linear_response_constrained(
        xy,
        keys,
        values,
        E_bulk,
        boundary_mask,
    )

    shear_energy = relaxed_spring_energy_linear_response_constrained(
        xy,
        keys,
        values,
        E_pure_shear,
        boundary_mask,
    )

    bulk_like = 2.0 * bulk_energy / (area * eps * eps + 1e-12)
    shear_like = 2.0 * shear_energy / (area * eps * eps + 1e-12)

    return {
        'linear_bulk_like': float(bulk_like),
        'linear_shear_like': float(shear_like),
        'linear_bulk_energy': float(bulk_energy),
        'linear_shear_energy': float(shear_energy),
        'linear_ratio_g_over_b': float(shear_like / (bulk_like + 1e-12)),
        'linear_log_g_over_b': float(
            np.log(max(shear_like, 1e-12))
            - np.log(max(bulk_like, 1e-12))
        ),
    }


def active_degrees(values, keys):
    threshold = float(cfg['mcmc_active_threshold'])
    degrees = {}

    for edge_idx, (i, j) in enumerate(keys):
        i, j = int(i), int(j)

        degrees.setdefault(i, 0)
        degrees.setdefault(j, 0)

        if values[edge_idx] > threshold:
            degrees[i] += 1
            degrees[j] += 1

    return degrees


def active_graph_connected(values, keys, n_nodes):
    threshold = float(cfg['mcmc_active_threshold'])

    adj = [[] for _ in range(n_nodes)]
    active_edge_count = 0

    for edge_idx, (i, j) in enumerate(keys):
        if values[edge_idx] > threshold:
            i, j = int(i), int(j)
            adj[i].append(j)
            adj[j].append(i)
            active_edge_count += 1

    if active_edge_count == 0:
        return False

    start = None

    for node_idx in range(n_nodes):
        if len(adj[node_idx]) > 0:
            start = node_idx
            break

    if start is None:
        return False

    seen = {start}
    stack = [start]

    while stack:
        node = stack.pop()

        for nb in adj[node]:
            if nb not in seen:
                seen.add(nb)
                stack.append(nb)

    active_nodes = {
        node_idx
        for node_idx in range(n_nodes)
        if len(adj[node_idx]) > 0
    }

    return seen == active_nodes


def physical_constraint_report(values, keys, n_nodes):
    values = np.asarray(values, dtype=float)

    threshold = float(cfg['mcmc_active_threshold'])
    near_zero_frac = float(np.mean(values <= threshold))

    degrees = active_degrees(values, keys)

    min_active_degree = min(degrees.values()) if degrees else 0
    low_degree_count = sum(
        1 for degree in degrees.values()
        if degree < int(cfg['rev_low_degree_min'])
    )

    connected = active_graph_connected(values, keys, n_nodes)

    return {
        'near_zero_frac': near_zero_frac,
        'min_active_degree': int(min_active_degree),
        'low_degree_count': int(low_degree_count),
        'active_connected': bool(connected),
    }


def prune_move_allowed(trial_values, keys, edge_idx, n_nodes):
    i, j = keys[int(edge_idx)]

    degrees_after = active_degrees(trial_values, keys)
    min_needed = int(cfg['rev_min_active_degree_after_prune'])

    if degrees_after.get(i, 0) < min_needed:
        return False, 'endpoint_i_low_active_degree'

    if degrees_after.get(j, 0) < min_needed:
        return False, 'endpoint_j_low_active_degree'

    report = physical_constraint_report(trial_values, keys, n_nodes)

    if report['near_zero_frac'] > float(cfg['rev_max_near_zero_frac']):
        return False, 'too_many_near_zero_edges'

    if cfg.get('rev_require_active_connected', False):
        if not report['active_connected']:
            return False, 'active_graph_disconnected'

    return True, 'ok'


def response_change_for_trial(current_lr, trial_lr):
    B0 = float(current_lr['linear_bulk_like'])
    G0 = float(current_lr['linear_shear_like'])

    B1 = float(trial_lr['linear_bulk_like'])
    G1 = float(trial_lr['linear_shear_like'])

    delta_B = B0 - B1
    delta_G = G0 - G1

    rel_delta_B = delta_B / (abs(B0) + 1e-12)
    rel_delta_G = delta_G / (abs(G0) + 1e-12)

    return {
        'rel_delta_B': float(rel_delta_B),
        'rel_delta_G': float(rel_delta_G),
        'B_after': float(B1),
        'G_after': float(G1),
        'G_over_B_after': float(G1 / (B1 + 1e-12)),
    }


def softmax_sample(items, score_key, temperature, rng):
    scores = np.asarray([item[score_key] for item in items], dtype=float)
    scores = scores - np.max(scores)

    temperature = max(float(temperature), 1e-12)

    weights = np.exp(scores / temperature)
    weights = weights / weights.sum()

    idx = int(rng.choice(np.arange(len(items)), p=weights))

    return items[idx]


def sample_edge_indices(values, count, rng, only_modified=False, reference_values=None):
    values = np.asarray(values, dtype=float)
    weights = np.ones(len(values), dtype=float)

    if only_modified:
        if reference_values is None:
            return []

        ref = np.asarray(reference_values, dtype=float)

        modified = np.abs(
            np.log((values + 1e-12) / (ref + 1e-12))
        ) > 1e-6

        weights[~modified] = 0.0

    movable = values > float(cfg['min_stiffness'])
    weights[~movable] = 0.0

    nonzero = np.flatnonzero(weights > 0)

    if len(nonzero) == 0:
        return []

    size = min(int(count), len(nonzero))

    weights = weights / weights.sum()

    return list(rng.choice(
        np.arange(len(values)),
        size=size,
        replace=False,
        p=weights,
    ))


def make_edge_trial(values, edge_idx, move_type, reference_values, rng):
    trial = np.asarray(values, dtype=float).copy()
    old_value = float(trial[edge_idx])

    if move_type == 'weaken':
        if rng.random() < float(cfg['rev_prob_prune_in_weaken_move']):
            new_value = float(cfg['rev_near_zero_value'])
            actual_move_type = 'prune'
        else:
            new_value = old_value * float(cfg['rev_weaken_factor'])
            actual_move_type = 'weaken'

    elif move_type == 'strengthen':
        new_value = old_value * float(cfg['rev_strengthen_factor'])
        actual_move_type = 'strengthen'

    elif move_type == 'restore':
        ref_value = float(reference_values[edge_idx])
        mix = float(cfg['rev_restore_mix'])
        new_value = (1.0 - mix) * old_value + mix * ref_value
        actual_move_type = 'restore'

    else:
        raise ValueError(f'Unknown move_type: {move_type}')

    new_value = float(np.clip(
        new_value,
        float(cfg['min_stiffness']),
        float(cfg['max_stiffness']),
    ))

    trial[edge_idx] = new_value

    return trial, actual_move_type, old_value, new_value


def propose_random_lammps_move(values, keys, graph, reference_values, rng):
    values = np.asarray(values, dtype=float)
    reference_values = np.asarray(reference_values, dtype=float)
    n_nodes = int(graph.x.shape[0])

    probs = np.asarray([
        cfg['rev_prob_weaken'],
        cfg['rev_prob_strengthen'],
        cfg['rev_prob_restore'],
    ], dtype=float)
    probs = probs / probs.sum()
    base_move_type = str(rng.choice(['weaken', 'strengthen', 'restore'], p=probs))

    for _ in range(max(10, int(cfg['rev_candidate_count']))):
        if base_move_type == 'restore':
            edge_indices = sample_edge_indices(
                values, 1, rng, only_modified=True, reference_values=reference_values
            )
            if not edge_indices:
                base_move_type = str(rng.choice(['weaken', 'strengthen']))
                edge_indices = sample_edge_indices(values, 1, rng)
        else:
            edge_indices = sample_edge_indices(values, 1, rng)

        if not edge_indices:
            return _empty_proposal(values, 'no_random_candidate_edges')

        edge_idx = int(edge_indices[0])
        trial, actual_move_type, old_value, new_value = make_edge_trial(
            values, edge_idx, base_move_type, reference_values, rng
        )

        if actual_move_type == 'prune':
            ok, reason = prune_move_allowed(trial, keys, edge_idx, n_nodes)
            if not ok:
                continue

        report = physical_constraint_report(trial, keys, n_nodes)
        if report['near_zero_frac'] > float(cfg['rev_max_near_zero_frac']):
            continue
        if cfg.get('rev_require_active_connected', False) and not report['active_connected']:
            continue

        return {
            'proposal_values': trial,
            'changed_edge_idx': edge_idx,
            'paired_edge_idx': None,
            'move_type': f'random_{actual_move_type}',
            'move_factor': float(new_value / (old_value + 1e-12)),
            'old_edge_value': float(old_value),
            'new_edge_value': float(new_value),
            'old_pair_edge_value': np.nan,
            'new_pair_edge_value': np.nan,
            'proposal_valid': True,
            'proposal_reject_reason': 'ok',
            'proposal_bias_score': np.nan,
            'rel_delta_B': np.nan,
            'rel_delta_G': np.nan,
            'B_after': np.nan,
            'G_after': np.nan,
            'G_over_B_after': np.nan,
        }

    return _empty_proposal(values, f'no_valid_random_{base_move_type}_candidate')


def _empty_proposal(values, reason):
    return {
        'proposal_values': np.asarray(values, dtype=float).copy(),
        'changed_edge_idx': None,
        'paired_edge_idx': None,
        'move_type': 'none',
        'move_factor': 1.0,
        'old_edge_value': np.nan,
        'new_edge_value': np.nan,
        'old_pair_edge_value': np.nan,
        'new_pair_edge_value': np.nan,
        'proposal_valid': False,
        'proposal_reject_reason': reason,
    }


def fit_low_pratio_target_distribution(summary_df):
    feature_cols = list(cfg['rev_target_feature_cols'])
    cutoff = float(cfg['rev_target_p_ratio_cutoff'])
    target = summary_df[summary_df['final_p_ratio'] < cutoff].copy()
    source = f"final_p_ratio < {cutoff:g}"

    if len(target) < int(cfg['rev_min_target_count']):
        q = float(cfg['rev_target_quantile_fallback'])
        cutoff = float(summary_df['final_p_ratio'].quantile(q))
        target = summary_df[summary_df['final_p_ratio'] <= cutoff].copy()
        source = f"lowest {q:g} quantile, final_p_ratio <= {cutoff:.4g}"

    target = target[['sim_idx', 'final_p_ratio', *feature_cols]].replace([np.inf, -np.inf], np.nan).dropna()

    if len(target) < 2:
        raise ValueError('Not enough valid low-p-ratio rows to fit the reversible-MCMC target distribution.')

    mu = target[feature_cols].mean().to_numpy(dtype=float)
    sigma = target[feature_cols].std(ddof=0).to_numpy(dtype=float)
    sigma = np.maximum(sigma, 1e-6)

    return {
        'feature_cols': feature_cols,
        'mu': mu,
        'sigma': sigma,
        'source': source,
        'count': int(len(target)),
        'target_rows': target,
    }


def target_distribution_log_prob(info, target_model):
    if target_model is None:
        return 0.0

    x = np.asarray([float(info[col]) for col in target_model['feature_cols']], dtype=float)
    z = (x - target_model['mu']) / target_model['sigma']

    return float(-0.5 * np.sum(z * z) - np.sum(np.log(target_model['sigma'])))


def reversible_mcmc_score(graph, keys, values, reference_values, target_model=None):
    values = np.asarray(values, dtype=float)
    reference_values = np.asarray(reference_values, dtype=float)

    lr = linear_response_moduli_for_values(graph, keys, values)
    reference_lr = linear_response_moduli_for_values(graph, keys, reference_values)

    B = max(float(lr['linear_bulk_like']), 1e-12)
    G = max(float(lr['linear_shear_like']), 1e-12)
    G_ref = max(float(reference_lr['linear_shear_like']), 1e-12)
    log_g_over_b = float(np.log(G) - np.log(B))
    shear_retention = float(G / G_ref)

    target_log_prob = target_distribution_log_prob(lr, target_model)
    score = (
        float(cfg['rev_log_g_over_b_weight']) * log_g_over_b
        + float(cfg['rev_target_distribution_weight']) * target_log_prob
    )

    n_nodes = int(graph.x.shape[0])
    report = physical_constraint_report(values, keys, n_nodes)

    near_zero_excess = max(
        0.0,
        report['near_zero_frac'] - float(cfg['rev_max_near_zero_frac']),
    )
    shear_retention_deficit = max(
        0.0,
        float(cfg['rev_min_shear_retention']) - shear_retention,
    )

    log_drift = np.log((values + 1e-12) / (reference_values + 1e-12))
    drift_penalty = float(np.mean(log_drift ** 2))

    low_degree_penalty = float(report['low_degree_count'])

    score -= float(cfg['rev_near_zero_penalty']) * near_zero_excess ** 2
    score -= float(cfg['rev_shear_retention_penalty']) * shear_retention_deficit ** 2
    score -= float(cfg['rev_log_stiffness_drift_penalty']) * drift_penalty
    score -= float(cfg['rev_low_degree_penalty']) * low_degree_penalty

    return float(score), {
        **lr,
        **report,
        'reversible_mcmc_score': float(score),
        'log_g_over_b_objective': float(log_g_over_b),
        'target_distribution_log_prob': float(target_log_prob),
        'shear_retention_vs_reference': float(shear_retention),
        'shear_retention_deficit': float(shear_retention_deficit),
        'log_stiffness_drift_penalty_raw': float(drift_penalty),
    }


def metropolis_accept(delta, temperature, rng):
    temperature = float(temperature)

    if delta >= 0:
        return True, 1.0

    log_accept = float(delta / temperature)
    log_u = float(np.log(rng.random()))

    accepted = bool(log_u < log_accept)
    accept_prob = float(np.exp(log_accept))

    return accepted, accept_prob


def p_ratio_at_frame(sim, frame_idx):
    side_idx = directional_side_indices_from_box(
        sim[0],
        quantile=cfg['side_quantile'],
    )

    frame_idx = min(int(frame_idx), len(sim) - 1)

    return float(calc_p_ratio_rollout_sides(
        sim,
        frame_idx,
        side_idx=side_idx,
    ))


def build_summary_from_data(data):
    rows = []

    for sim_idx, sim in enumerate(data):
        graph0 = sim[0]

        keys, values = unique_edge_stiffness(graph0)
        lr = linear_response_moduli_for_values(graph0, keys, values)

        row = {
            'sim_idx': int(sim_idx),
            'frames': int(len(sim)),
            'nodes': int(graph0.x.shape[0]),
            'edges_unique': int(values.shape[0]),
            'final_p_ratio': p_ratio_at_frame(sim, cfg['target_frame']),
            **lr,
        }

        rows.append(row)

    return pd.DataFrame(rows)


def poisson_ratio_from_g_over_b(g_over_b):
    r = float(g_over_b)
    if not np.isfinite(r) or r <= 0:
        return np.nan
    return float((1.0 - r) / (1.0 + r))


def metaforge_score_from_result(result, reference_result=None):
    bulk = float(result.get('bulk_modulus', np.nan))
    shear = float(result.get('shear_modulus', np.nan))
    metaforge_p_ratio = float(result.get('metaforge_p_ratio', np.nan))

    if reference_result is None:
        bulk_retention = 1.0
        shear_retention = 1.0
    else:
        reference_bulk = max(float(reference_result.get('bulk_modulus', np.nan)), 1e-12)
        reference_shear = max(float(reference_result.get('shear_modulus', np.nan)), 1e-12)
        bulk_retention = float(bulk / reference_bulk) if np.isfinite(bulk) else 0.0
        shear_retention = float(shear / reference_shear) if np.isfinite(shear) else 0.0

    bulk_retention_deficit = max(0.0, float(cfg['valid_min_bulk_retention']) - bulk_retention)
    shear_retention_deficit = max(0.0, float(cfg['valid_min_shear_retention']) - shear_retention)
    shear_objective_deficit = max(0.0, float(cfg['rev_lammps_min_shear_retention']) - shear_retention)

    valid = (
        np.isfinite(metaforge_p_ratio)
        and np.isfinite(bulk)
        and np.isfinite(shear)
        and bulk > float(cfg['valid_bulk_min'])
        and shear > float(cfg['valid_shear_min'])
        and bulk_retention >= float(cfg['valid_min_bulk_retention'])
        and shear_retention >= float(cfg['valid_min_shear_retention'])
    )

    g_over_b = float(shear / (bulk + 1e-12)) if np.isfinite(bulk) and np.isfinite(shear) else np.nan
    log_g_over_b = float(np.log(max(g_over_b, 1e-12))) if np.isfinite(g_over_b) else -float(cfg['rev_lammps_invalid_penalty'])
    implied_p_ratio = poisson_ratio_from_g_over_b(g_over_b)
    p_ratio_formula_error = abs(metaforge_p_ratio - implied_p_ratio) if np.isfinite(metaforge_p_ratio) and np.isfinite(implied_p_ratio) else np.nan

    # For this elastic convention, nu = (1 - G/B) / (1 + G/B), so maximizing
    # true LAMMPS log(G/B) is the same as minimizing the reported p-ratio.
    score = log_g_over_b
    score -= float(cfg['rev_lammps_shear_retention_penalty']) * shear_objective_deficit ** 2
    if not valid:
        score -= float(cfg['rev_lammps_invalid_penalty'])

    return float(score), {
        'metaforge_score': float(score),
        'metaforge_p_ratio': metaforge_p_ratio,
        'metaforge_implied_p_ratio': implied_p_ratio,
        'metaforge_p_ratio_formula_error': float(p_ratio_formula_error) if np.isfinite(p_ratio_formula_error) else np.nan,
        'bulk_modulus': bulk,
        'shear_modulus': shear,
        'metaforge_g_over_b': g_over_b,
        'metaforge_log_g_over_b': log_g_over_b,
        'metaforge_bulk_retention': float(bulk_retention),
        'metaforge_shear_retention': float(shear_retention),
        'metaforge_bulk_retention_deficit': float(bulk_retention_deficit),
        'metaforge_shear_retention_deficit': float(shear_retention_deficit),
        'metaforge_shear_objective_deficit': float(shear_objective_deficit),
        'mechanically_valid': bool(valid),
    }


def lammps_score_for_values(start_graph, keys, values, variant, cache, cache_path, reference_result=None):
    signature = repr(stiffness_signature(values))
    if signature in cache:
        cached = dict(cache[signature])
        score, info = metaforge_score_from_result(cached, reference_result=reference_result)
        info.update({
            'stiffness_signature': signature,
            'lammps_cache_hit': True,
            'lammps_variant': variant,
        })
        return score, info

    graph = update_graph_stiffness(start_graph, keys, values)
    label = f'rev_lammps_mcmc_{variant}'[:180]
    sim_idx_for_label = int(str(variant).split('_')[0]) if str(variant).split('_')[0].isdigit() else 0
    result = run_metaforge_elastic(
        graph,
        label,
        start_graph=start_graph,
        keys=keys,
        values=values,
        start_sim_idx=sim_idx_for_label,
    )
    score, info = metaforge_score_from_result(result, reference_result=reference_result)
    out = {
        'stiffness_signature': signature,
        'lammps_cache_hit': False,
        **result,
        **info,
    }
    cache[signature] = dict(out)
    save_lammps_cache(cache_path, cache)
    info.update({
        'stiffness_signature': signature,
        'lammps_cache_hit': False,
        'lammps_variant': variant,
    })
    return score, info


def run_reversible_response_mcmc_for_graph(start_graph, start_sim_idx, rng, target_model=None):
    keys, start_values = unique_edge_stiffness(start_graph)
    edge_geometry_df = unique_edge_geometry(start_graph, keys)

    current_values = start_values.copy()

    if cfg['run_lammps'] and shutil.which(cfg['lammps_cmd']) is None:
        raise FileNotFoundError(f"LAMMPS command not found: {cfg['lammps_cmd']}")

    lammps_cache_path = output_dir / f'rev_lammps_in_loop_cache_sim_{start_sim_idx:03d}.csv'
    lammps_cache = load_lammps_cache(lammps_cache_path)

    current_score, lammps_info = lammps_score_for_values(
        start_graph, keys, current_values,
        f'{start_sim_idx:03d}_step_00000_original',
        lammps_cache, lammps_cache_path, reference_result=None,
    )
    reference_lammps_result = dict(lammps_info)
    _, approx_info = reversible_mcmc_score(
        start_graph, keys, current_values, reference_values=start_values, target_model=target_model
    )
    current_info = {**approx_info, **lammps_info, 'score_source': 'lammps'}

    records = []
    kept_value_rows = []

    best_values = current_values.copy()
    best_score = current_score
    best_info = dict(current_info)

    n_steps = int(cfg['rev_lammps_mcmc_steps'])
    burn_in = int(cfg['rev_lammps_burn_in'])
    keep_every = int(cfg['rev_lammps_keep_every'])
    temperature = float(cfg['rev_lammps_temperature'])

    for step in range(n_steps + 1):
        if step > 0:
            proposal = propose_random_lammps_move(
                current_values,
                keys,
                start_graph,
                start_values,
                rng,
            )

            if proposal['proposal_valid']:
                proposal_values = proposal['proposal_values']

                proposal_score, lammps_info = lammps_score_for_values(
                    start_graph, keys, proposal_values,
                    f'{start_sim_idx:03d}_step_{step:05d}_{proposal["move_type"]}',
                    lammps_cache, lammps_cache_path, reference_result=reference_lammps_result,
                )
                _, approx_info = reversible_mcmc_score(
                    start_graph, keys, proposal_values, reference_values=start_values, target_model=target_model
                )
                proposal_info = {**approx_info, **lammps_info, 'score_source': 'lammps'}

                delta = proposal_score - current_score

                accepted, accept_prob = metropolis_accept(
                    delta,
                    temperature,
                    rng,
                )

                if accepted:
                    current_values = proposal_values
                    current_score = proposal_score
                    current_info = proposal_info

                    if current_score > best_score:
                        best_score = current_score
                        best_values = current_values.copy()
                        best_info = dict(current_info)
            else:
                delta = np.nan
                accepted = False
                accept_prob = 0.0

        else:
            proposal = {
                'changed_edge_idx': None,
                'paired_edge_idx': None,
                'move_type': 'initial',
                'move_factor': 1.0,
                'old_edge_value': np.nan,
                'new_edge_value': np.nan,
                'old_pair_edge_value': np.nan,
                'new_pair_edge_value': np.nan,
                'proposal_valid': True,
                'proposal_reject_reason': 'ok',
                'proposal_bias_score': np.nan,
                'rel_delta_B': np.nan,
                'rel_delta_G': np.nan,
                'B_after': np.nan,
                'G_after': np.nan,
                'G_over_B_after': np.nan,
            }

            delta = 0.0
            accepted = True
            accept_prob = 1.0

        keep = (
            step >= burn_in
            and step % keep_every == 0
        )

        variant = f'rev_mcmc_step_{step:05d}' if keep else ''

        row = {
            'sim_idx': int(start_sim_idx),
            'step': int(step),
            'variant': variant,
            'accepted': bool(accepted),
            'accept_prob': float(accept_prob),
            'delta_score': float(delta) if np.isfinite(delta) else np.nan,
            'changed_edge_idx': proposal['changed_edge_idx'],
            'paired_edge_idx': proposal.get('paired_edge_idx'),
            'move_type': proposal['move_type'],
            'move_factor': float(proposal['move_factor']),
            'old_edge_value': float(proposal['old_edge_value']) if np.isfinite(proposal['old_edge_value']) else np.nan,
            'new_edge_value': float(proposal['new_edge_value']) if np.isfinite(proposal['new_edge_value']) else np.nan,
            'old_pair_edge_value': float(proposal.get('old_pair_edge_value', np.nan)) if np.isfinite(proposal.get('old_pair_edge_value', np.nan)) else np.nan,
            'new_pair_edge_value': float(proposal.get('new_pair_edge_value', np.nan)) if np.isfinite(proposal.get('new_pair_edge_value', np.nan)) else np.nan,
            'proposal_valid': bool(proposal['proposal_valid']),
            'proposal_reject_reason': proposal['proposal_reject_reason'],
            'proposal_bias_score': float(proposal.get('proposal_bias_score', np.nan)) if np.isfinite(proposal.get('proposal_bias_score', np.nan)) else np.nan,
            'rel_delta_B': float(proposal.get('rel_delta_B', np.nan)) if np.isfinite(proposal.get('rel_delta_B', np.nan)) else np.nan,
            'rel_delta_G': float(proposal.get('rel_delta_G', np.nan)) if np.isfinite(proposal.get('rel_delta_G', np.nan)) else np.nan,
            'B_after': float(proposal.get('B_after', np.nan)) if np.isfinite(proposal.get('B_after', np.nan)) else np.nan,
            'G_after': float(proposal.get('G_after', np.nan)) if np.isfinite(proposal.get('G_after', np.nan)) else np.nan,
            'G_over_B_after': float(proposal.get('G_over_B_after', np.nan)) if np.isfinite(proposal.get('G_over_B_after', np.nan)) else np.nan,
            'lammps_accept_score': float(current_score),
            'best_lammps_accept_score_so_far': float(best_score),
            **current_info,
        }

        records.append(row)

        if keep:
            kept_value_rows.append({
                'variant': variant,
                'values': current_values.copy(),
                'lammps_accept_score': float(current_score),
                **current_info,
            })

        if step % 50 == 0:
            print(
                f"step={step:5d} "
                f"score={current_score:9.5f} "
                f"true_G/B={current_info.get('metaforge_g_over_b', np.nan):.5g} "
                f"true_p={current_info.get('metaforge_p_ratio', np.nan):.5g} "
                f"B={current_info.get('bulk_modulus', np.nan):.4g} "
                f"G={current_info.get('shear_modulus', np.nan):.4g} "
                f"move={proposal['move_type']:10s} "
                f"accepted={accepted} "
                f"dB={proposal.get('rel_delta_B', np.nan): .3g} "
                f"dG={proposal.get('rel_delta_G', np.nan): .3g} "
                f"zero_frac={current_info['near_zero_frac']:.3f} "
                f"low_deg={current_info['low_degree_count']}"
            )

    special_values = {
        'original': start_values.copy(),
        'rev_mcmc_best_score': best_values.copy(),
    }

    trace_df = pd.DataFrame(records)

    return trace_df, kept_value_rows, special_values, keys, edge_geometry_df


def stiffness_signature(values, decimals=8):
    arr = np.asarray(values, dtype=float)
    return tuple(np.round(arr, int(decimals)).tolist())


def stiffness_log_distance(values_a, values_b):
    a = np.asarray(values_a, dtype=float)
    b = np.asarray(values_b, dtype=float)
    return float(np.sqrt(np.mean((np.log(a + 1e-12) - np.log(b + 1e-12)) ** 2)))


def select_lammps_variants(candidate_df, values_by_variant):
    required = [variant for variant in ['original', 'rev_mcmc_best_score'] if variant in values_by_variant]
    budget = max(0, int(cfg['rev_lammps_budget']))
    prefilter = max(budget, int(cfg['rev_lammps_prefilter_count']))
    min_dist = float(cfg['rev_lammps_diversity_min_logdist'])

    selected = []
    seen_signatures = set()

    def add_variant(variant):
        if variant not in values_by_variant:
            return False
        sig = stiffness_signature(values_by_variant[variant])
        if sig in seen_signatures:
            return False
        selected.append(variant)
        seen_signatures.add(sig)
        return True

    for variant in required:
        add_variant(variant)

    pool = candidate_df.head(prefilter).copy()
    for _, row in pool.iterrows():
        if len(selected) >= budget + len(required):
            break
        variant = row['variant']
        if variant in selected:
            continue
        values = values_by_variant.get(variant)
        if values is None:
            continue
        if selected and min(
            stiffness_log_distance(values, values_by_variant[old_variant])
            for old_variant in selected
        ) < min_dist:
            continue
        add_variant(variant)

    # If the diversity filter was too strict, fill remaining budget by rank.
    for _, row in pool.iterrows():
        if len(selected) >= budget + len(required):
            break
        add_variant(row['variant'])

    return selected


def load_lammps_cache(cache_path):
    if cache_path.exists():
        cache_df = pd.read_csv(cache_path)
        if 'stiffness_signature' in cache_df.columns:
            return {
                row['stiffness_signature']: row.to_dict()
                for _, row in cache_df.iterrows()
            }
    return {}


def save_lammps_cache(cache_path, cache):
    if cache:
        pd.DataFrame(cache.values()).to_csv(cache_path, index=False)


def metaforge_elastic_config(*, fast_screening=None):
    return ElasticRunConfig(
        lammps_cmd=str(cfg['lammps_cmd']),
        mass=float(cfg['mass']),
        angles=float(cfg['angles']),
        screen=cfg.get('lammps_screen', None),
        fast_screening=bool(cfg.get('lammps_fast_screening', True) if fast_screening is None else fast_screening),
        fast=FastElasticSettings(
            maxiter=int(cfg['lammps_fast_maxiter']),
            maxeval=int(cfg['lammps_fast_maxeval']),
            ftol=float(cfg['lammps_fast_ftol']),
            dmax=float(cfg['lammps_fast_dmax']),
            srate=float(cfg['lammps_fast_srate']),
        ),
        reuse_topology=bool(cfg.get('lammps_reuse_network_topology', True)),
    )


def run_metaforge_elastic(graph, label, *, start_graph=None, keys=None, values=None, start_sim_idx=None):
    can_reuse_topology = (
        cfg.get('lammps_reuse_network_topology', True)
        and start_graph is not None
        and values is not None
        and start_sim_idx is not None
    )
    return run_elastic_simulation(
        graph,
        parent_dir=run_dir,
        label=label,
        config=metaforge_elastic_config(),
        base_graph=start_graph if can_reuse_topology else None,
        base_cache_key=f'{int(start_sim_idx):03d}' if can_reuse_topology else None,
        bond_stiffnesses=np.asarray(values, dtype=float) if can_reuse_topology else None,
        box=start_graph.box if can_reuse_topology else graph.box,
    )


data = torch.load(
    cfg['dataset_path'],
    map_location='cpu',
    weights_only=False,
)

print('trajectories:', len(data))
print('frames per first trajectory:', len(data[0]))
print(
    'nodes:',
    data[0][0].x.shape[0],
    'directed edges:',
    data[0][0].edge_index.shape[1],
)

summary_path = output_dir / 'reid_reversible_response_mcmc_summary.csv'

summary_required_cols = {'sim_idx', 'final_p_ratio', *cfg['rev_target_feature_cols']}

if summary_path.exists():
    summary_df = pd.read_csv(summary_path)
else:
    summary_df = pd.DataFrame()

if summary_df.empty or not summary_required_cols.issubset(summary_df.columns):
    summary_df = build_summary_from_data(data)
    summary_df.to_csv(summary_path, index=False)

print(summary_df['final_p_ratio'].describe().to_string())

target_model = fit_low_pratio_target_distribution(summary_df)
print('target distribution source:', target_model['source'])
print('target distribution count:', target_model['count'])
print('target features:', target_model['feature_cols'])
print('target mu:', dict(zip(target_model['feature_cols'], target_model['mu'])))
print('target sigma:', dict(zip(target_model['feature_cols'], target_model['sigma'])))

if cfg['rev_start_sim_idx'] is None:
    start_sim_idx = int(
        summary_df.sort_values(
            'final_p_ratio',
            ascending=False,
        ).iloc[0]['sim_idx']
    )
else:
    start_sim_idx = int(cfg['rev_start_sim_idx'])

start_graph = data[start_sim_idx][0]

print('starting sim_idx:', start_sim_idx)
print(
    'starting rollout p-ratio:',
    float(summary_df.loc[
        summary_df['sim_idx'].eq(start_sim_idx),
        'final_p_ratio',
    ].iloc[0]),
)

orig_keys, orig_values = unique_edge_stiffness(start_graph)
orig_lr = linear_response_moduli_for_values(start_graph, orig_keys, orig_values)

print('original approximate B:', orig_lr['linear_bulk_like'])
print('original approximate G:', orig_lr['linear_shear_like'])
print('original approximate G/B:', orig_lr['linear_ratio_g_over_b'])

trace_df, kept_value_rows, special_values, keys, edge_geometry_df = run_reversible_response_mcmc_for_graph(
    start_graph=start_graph,
    start_sim_idx=start_sim_idx,
    rng=rng,
    target_model=target_model,
)

def summarize_reversible_mcmc_trace(trace_df, output_dir, start_sim_idx):
    trace_df = trace_df.copy()
    non_initial = trace_df[trace_df['step'].gt(0)].copy()

    if 'linear_ratio_g_over_b' in trace_df.columns:
        trace_df['linear_implied_p_ratio'] = trace_df['linear_ratio_g_over_b'].map(poisson_ratio_from_g_over_b)
        trace_df['best_linear_implied_p_ratio_so_far'] = trace_df['linear_implied_p_ratio'].cummin()

    if 'metaforge_p_ratio' in trace_df.columns:
        trace_df['best_true_p_ratio_so_far'] = trace_df['metaforge_p_ratio'].cummin()

    if 'metaforge_g_over_b' in trace_df.columns:
        trace_df['best_true_g_over_b_so_far'] = trace_df['metaforge_g_over_b'].cummax()

    original_row = trace_df.iloc[0]
    original_true_p = float(original_row.get('metaforge_p_ratio', np.nan))
    original_linear_p = float(original_row.get('linear_implied_p_ratio', np.nan))

    summary_rows = []
    attempted = int(len(non_initial))
    valid_count = int(non_initial['proposal_valid'].sum()) if 'proposal_valid' in non_initial else 0
    accepted_count = int(non_initial['accepted'].sum()) if 'accepted' in non_initial else 0
    accepted_valid_count = int((non_initial['proposal_valid'] & non_initial['accepted']).sum()) if len(non_initial) else 0
    cache_hit_count = int(non_initial.get('lammps_cache_hit', pd.Series(False, index=non_initial.index)).fillna(False).sum())
    true_lammps_runs = int(valid_count - cache_hit_count)

    best_true_p = float(trace_df['metaforge_p_ratio'].min()) if 'metaforge_p_ratio' in trace_df else np.nan
    final_true_p = float(trace_df['metaforge_p_ratio'].iloc[-1]) if 'metaforge_p_ratio' in trace_df else np.nan
    best_linear_p = float(trace_df['linear_implied_p_ratio'].min()) if 'linear_implied_p_ratio' in trace_df else np.nan
    final_linear_p = float(trace_df['linear_implied_p_ratio'].iloc[-1]) if 'linear_implied_p_ratio' in trace_df else np.nan

    summary_rows.append({
        'scope': 'overall',
        'attempted_steps': attempted,
        'valid_proposals': valid_count,
        'accepted_steps': accepted_count,
        'accepted_valid_steps': accepted_valid_count,
        'invalid_proposals': int(attempted - valid_count),
        'lammps_cache_hits': cache_hit_count,
        'true_lammps_runs': true_lammps_runs,
        'valid_rate': valid_count / attempted if attempted else np.nan,
        'accept_rate': accepted_count / attempted if attempted else np.nan,
        'accept_rate_given_valid': accepted_valid_count / valid_count if valid_count else np.nan,
        'cache_hit_rate_given_valid': cache_hit_count / valid_count if valid_count else np.nan,
        'original_true_p_ratio': original_true_p,
        'final_true_p_ratio': final_true_p,
        'best_true_p_ratio': best_true_p,
        'best_true_p_ratio_improvement': original_true_p - best_true_p if np.isfinite(original_true_p) and np.isfinite(best_true_p) else np.nan,
        'original_linear_implied_p_ratio': original_linear_p,
        'final_linear_implied_p_ratio': final_linear_p,
        'best_linear_implied_p_ratio': best_linear_p,
        'best_linear_p_ratio_improvement': original_linear_p - best_linear_p if np.isfinite(original_linear_p) and np.isfinite(best_linear_p) else np.nan,
    })

    overall_summary_df = pd.DataFrame(summary_rows)

    if len(non_initial):
        move_summary_df = (
            non_initial
            .groupby('move_type', dropna=False)
            .agg(
                steps=('step', 'count'),
                valid_proposals=('proposal_valid', 'sum'),
                accepted_steps=('accepted', 'sum'),
                mean_accept_prob=('accept_prob', 'mean'),
                median_delta_score=('delta_score', 'median'),
                best_true_p_ratio=('metaforge_p_ratio', 'min'),
                final_true_p_ratio=('metaforge_p_ratio', 'last'),
            )
            .reset_index()
        )
        move_summary_df['valid_rate'] = move_summary_df['valid_proposals'] / move_summary_df['steps']
        move_summary_df['accept_rate'] = move_summary_df['accepted_steps'] / move_summary_df['steps']
    else:
        move_summary_df = pd.DataFrame()

    window = 100
    if len(non_initial):
        window_df = non_initial.copy()
        window_df['step_window'] = ((window_df['step'] - 1) // window) * window + 1
        window_summary_df = (
            window_df
            .groupby('step_window', dropna=False)
            .agg(
                steps=('step', 'count'),
                valid_proposals=('proposal_valid', 'sum'),
                accepted_steps=('accepted', 'sum'),
                true_lammps_cache_hits=('lammps_cache_hit', 'sum'),
                mean_accept_prob=('accept_prob', 'mean'),
                mean_delta_score=('delta_score', 'mean'),
                best_true_p_ratio=('metaforge_p_ratio', 'min'),
                final_true_p_ratio=('metaforge_p_ratio', 'last'),
                best_true_g_over_b=('metaforge_g_over_b', 'max'),
            )
            .reset_index()
        )
        window_summary_df['valid_rate'] = window_summary_df['valid_proposals'] / window_summary_df['steps']
        window_summary_df['accept_rate'] = window_summary_df['accepted_steps'] / window_summary_df['steps']
    else:
        window_summary_df = pd.DataFrame()

    summary_path = output_dir / f'rev_response_mcmc_success_summary_sim_{start_sim_idx:03d}.csv'
    move_path = output_dir / f'rev_response_mcmc_success_by_move_sim_{start_sim_idx:03d}.csv'
    window_path = output_dir / f'rev_response_mcmc_success_by_step_window_sim_{start_sim_idx:03d}.csv'
    overall_summary_df.to_csv(summary_path, index=False)
    move_summary_df.to_csv(move_path, index=False)
    window_summary_df.to_csv(window_path, index=False)

    print('MCMC success summary')
    print(overall_summary_df.round(6).to_string(index=False))
    if len(move_summary_df):
        print('\nBy move type')
        print(move_summary_df.sort_values('steps', ascending=False).round(6).to_string(index=False))
    if len(window_summary_df):
        print('\nBy 100-step window')
        print(window_summary_df.round(6).to_string(index=False))

    fig, axes = plt.subplots(2, 2, figsize=(11.5, 7.0), constrained_layout=True)

    ax = axes[0, 0]
    ax.plot(trace_df['step'], trace_df['metaforge_p_ratio'], color=PAPER_COLORS['blue'], lw=1.2, alpha=0.75, label='current true p')
    if 'best_true_p_ratio_so_far' in trace_df:
        ax.plot(trace_df['step'], trace_df['best_true_p_ratio_so_far'], color=PAPER_COLORS['red'], lw=1.6, label='best true p')
    ax.axhline(original_true_p, color='#333333', lw=1, ls='--', label='original')
    ax.set_xlabel('MCMC step')
    ax.set_ylabel('LAMMPS p-ratio')
    ax.set_title('True p-ratio progress')
    ax.legend(frameon=False, fontsize=8)
    style_axes(ax)

    ax = axes[0, 1]
    if 'linear_implied_p_ratio' in trace_df:
        ax.plot(trace_df['step'], trace_df['linear_implied_p_ratio'], color=PAPER_COLORS['green'], lw=1.1, alpha=0.75, label='current linear-implied p')
        ax.plot(trace_df['step'], trace_df['best_linear_implied_p_ratio_so_far'], color=PAPER_COLORS['orange'], lw=1.5, label='best linear-implied p')
        ax.axhline(original_linear_p, color='#333333', lw=1, ls='--', label='original')
    ax.set_xlabel('MCMC step')
    ax.set_ylabel('predicted/implied p-ratio')
    ax.set_title('Approximate p-ratio progress')
    ax.legend(frameon=False, fontsize=8)
    style_axes(ax)

    ax = axes[1, 0]
    if len(window_summary_df):
        ax.plot(window_summary_df['step_window'], window_summary_df['accept_rate'], marker='o', ms=3, lw=1.2, label='accepted / attempted')
        ax.plot(window_summary_df['step_window'], window_summary_df['valid_rate'], marker='o', ms=3, lw=1.2, label='valid / attempted')
    ax.set_xlabel('MCMC step window')
    ax.set_ylabel('rate')
    ax.set_ylim(-0.03, 1.03)
    ax.set_title('Success rates by window')
    ax.legend(frameon=False, fontsize=8)
    style_axes(ax)

    ax = axes[1, 1]
    if len(move_summary_df):
        x = np.arange(len(move_summary_df))
        ordered = move_summary_df.sort_values('steps', ascending=False).reset_index(drop=True)
        ax.bar(x - 0.18, ordered['valid_rate'], width=0.36, label='valid')
        ax.bar(x + 0.18, ordered['accept_rate'], width=0.36, label='accepted')
        ax.set_xticks(x)
        ax.set_xticklabels(ordered['move_type'], rotation=30, ha='right')
    ax.set_ylabel('rate')
    ax.set_ylim(-0.03, 1.03)
    ax.set_title('Success rates by move type')
    ax.legend(frameon=False, fontsize=8)
    style_axes(ax)

    diagnostics_plot_path = output_dir / f'rev_response_mcmc_success_diagnostics_sim_{start_sim_idx:03d}.png'
    fig.savefig(diagnostics_plot_path, dpi=180)
    plt.show()
    print('saved MCMC diagnostics:', diagnostics_plot_path)

    return trace_df, overall_summary_df, move_summary_df, window_summary_df


trace_df, mcmc_success_summary_df, mcmc_move_summary_df, mcmc_window_summary_df = summarize_reversible_mcmc_trace(
    trace_df,
    output_dir,
    start_sim_idx,
)

trace_path = output_dir / f'rev_response_mcmc_trace_sim_{start_sim_idx:03d}.csv'
trace_df.to_csv(trace_path, index=False)

candidate_rows = []

for item in kept_value_rows:
    row = {
        'sim_idx': start_sim_idx,
        'variant': item['variant'],
        **{
            key: value
            for key, value in item.items()
            if key not in ['variant', 'values']
        },
    }

    candidate_rows.append(row)

for variant, values in special_values.items():
    score, info = reversible_mcmc_score(
        start_graph,
        keys,
        values,
        reference_values=special_values['original'],
        target_model=target_model,
    )

    row = {
        'sim_idx': start_sim_idx,
        'variant': variant,
        'reversible_mcmc_score': float(score),
        **info,
    }

    candidate_rows.append(row)

candidate_df = pd.DataFrame(candidate_rows)

candidate_sort_cols = [
    col for col in [
        'lammps_accept_score',
        'metaforge_score',
        'metaforge_g_over_b',
        'reversible_mcmc_score',
        'linear_ratio_g_over_b',
    ]
    if col in candidate_df.columns
]
candidate_df = candidate_df.sort_values(
    candidate_sort_cols,
    ascending=[False] * len(candidate_sort_cols),
    na_position='last',
).drop_duplicates('variant').reset_index(drop=True)

candidate_path = output_dir / f'rev_response_mcmc_candidates_sim_{start_sim_idx:03d}.csv'
candidate_df.to_csv(candidate_path, index=False)

print('saved trace:', trace_path)
print('saved candidates:', candidate_path)

display(candidate_df.head(20))

values_by_variant = {}

for item in kept_value_rows:
    values_by_variant[item['variant']] = item['values']

for variant, values in special_values.items():
    values_by_variant[variant] = values

if cfg['run_lammps'] and shutil.which(cfg['lammps_cmd']) is None:
    raise FileNotFoundError(f"LAMMPS command not found: {cfg['lammps_cmd']}")

lammps_rows = []
final_reference_result = None
final_fast_screening = bool(cfg.get('final_lammps_fast_screening', cfg.get('lammps_fast_screening', True)))
final_lammps_mode = 'fast' if final_fast_screening else 'full'
lammps_cache_path = output_dir / f'rev_response_mcmc_lammps_{final_lammps_mode}_cache_sim_{start_sim_idx:03d}.csv'
lammps_cache = load_lammps_cache(lammps_cache_path)
previous_lammps_fast_screening = cfg.get('lammps_fast_screening', True)
cfg['lammps_fast_screening'] = final_fast_screening

if cfg['run_lammps']:
    print(f'final LAMMPS validation mode: {final_lammps_mode}')
    top_variants = select_lammps_variants(candidate_df, values_by_variant)
    print(
        f'LAMMPS budget selected {len(top_variants)} variants '
        f'from {min(len(candidate_df), int(cfg["rev_lammps_prefilter_count"]))} prefiltered candidates.'
    )

    for variant in top_variants:
        values = values_by_variant[variant]
        signature = repr(stiffness_signature(values))

        if signature in lammps_cache:
            cached = dict(lammps_cache[signature])
            out = {**cached, 'sim_idx': start_sim_idx, 'variant': variant, 'lammps_cache_hit': True, 'lammps_validation_mode': final_lammps_mode}
            _, validity_info = metaforge_score_from_result(out, reference_result=final_reference_result)
            out.update(validity_info)
            if variant == 'original':
                final_reference_result = dict(out)
            print('cache hit', variant, 'valid=', out['mechanically_valid'])
            lammps_rows.append(out)
            continue

        graph = update_graph_stiffness(start_graph, keys, values)
        label = f'rev_mcmc_{start_sim_idx:03d}_{variant}'[:180]
        print('running', label)

        try:
            result = run_metaforge_elastic(
                graph,
                label,
                start_graph=start_graph,
                keys=keys,
                values=values,
                start_sim_idx=start_sim_idx,
            )
            out = {
                'sim_idx': start_sim_idx,
                'variant': variant,
                'stiffness_signature': signature,
                'lammps_cache_hit': False,
                'lammps_validation_mode': final_lammps_mode,
                **result,
            }
            _, validity_info = metaforge_score_from_result(out, reference_result=final_reference_result)
            out.update(validity_info)
            if variant == 'original':
                final_reference_result = dict(out)
            lammps_cache[signature] = dict(out)
            save_lammps_cache(lammps_cache_path, lammps_cache)

            print(
                f"  p-ratio={out['metaforge_p_ratio']:.6g}, "
                f"G/B={out['metaforge_g_over_b']:.6g}, "
                f"bulk={out['bulk_modulus']:.6g}, "
                f"shear={out['shear_modulus']:.6g}, "
                f"bulk_ret={out.get('metaforge_bulk_retention', np.nan):.4g}, "
                f"shear_ret={out.get('metaforge_shear_retention', np.nan):.4g}, "
                f"valid={out['mechanically_valid']}"
            )

        except Exception as exc:
            out = {
                'sim_idx': start_sim_idx,
                'variant': variant,
                'stiffness_signature': signature,
                'lammps_cache_hit': False,
                'lammps_validation_mode': final_lammps_mode,
                'metaforge_p_ratio': np.nan,
                'bulk_modulus': np.nan,
                'shear_modulus': np.nan,
                'metaforge_g_over_b': np.nan,
                'metaforge_log_g_over_b': np.nan,
                'mechanically_valid': False,
                'error': repr(exc),
            }
            lammps_cache[signature] = dict(out)
            save_lammps_cache(lammps_cache_path, lammps_cache)
            print('  failed:', repr(exc))

        lammps_rows.append(out)

else:
    print('cfg[run_lammps] is False, skipping MetaForge validation.')

cfg['lammps_fast_screening'] = previous_lammps_fast_screening
lammps_df = pd.DataFrame(lammps_rows)

if not lammps_df.empty:
    lammps_result_cols = [
        'metaforge_p_ratio',
        'metaforge_implied_p_ratio',
        'metaforge_p_ratio_formula_error',
        'bulk_modulus',
        'shear_modulus',
        'metaforge_g_over_b',
        'metaforge_log_g_over_b',
        'metaforge_bulk_retention',
        'metaforge_shear_retention',
        'metaforge_bulk_retention_deficit',
        'metaforge_shear_retention_deficit',
        'metaforge_shear_objective_deficit',
        'mechanically_valid',
        'stdout_tail',
        'stderr_tail',
        'error',
    ]
    candidate_meta_df = candidate_df.drop(columns=lammps_result_cols, errors='ignore')
    result_df = candidate_meta_df.merge(
        lammps_df,
        on=['sim_idx', 'variant'],
        how='right',
    )
    result_df['metaforge_g_over_b'] = (
        result_df['shear_modulus']
        / (result_df['bulk_modulus'] + 1e-12)
    )
    result_df['metaforge_log_g_over_b'] = np.log(
        np.maximum(result_df['metaforge_g_over_b'], 1e-12)
    )
    result_df['metaforge_implied_p_ratio'] = result_df['metaforge_g_over_b'].map(poisson_ratio_from_g_over_b)
    result_df['metaforge_p_ratio_formula_error'] = (
        result_df['metaforge_p_ratio'] - result_df['metaforge_implied_p_ratio']
    ).abs()

    original_rows_for_validity = result_df[result_df['variant'].eq('original')]
    reference_for_validity = original_rows_for_validity.iloc[0].to_dict() if len(original_rows_for_validity) else None
    validity_rows = []
    for _, validity_row in result_df.iterrows():
        reference = None if validity_row['variant'] == 'original' else reference_for_validity
        _, validity_info = metaforge_score_from_result(validity_row.to_dict(), reference_result=reference)
        validity_rows.append(validity_info)
    if validity_rows:
        validity_df = pd.DataFrame(validity_rows)
        for col in validity_df.columns:
            result_df[col] = validity_df[col].to_numpy()

    result_path = output_dir / f'rev_response_mcmc_metaforge_results_sim_{start_sim_idx:03d}.csv'
    result_df.to_csv(result_path, index=False)

    print('saved MetaForge results:', result_path)

    valid = result_df[result_df['mechanically_valid'].fillna(False)].copy()

    if valid.empty:
        print('No mechanically valid reversible-MCMC candidate found.')
    else:
        best_row = valid.sort_values(
            ['metaforge_g_over_b', 'metaforge_p_ratio'],
            ascending=[False, True],
        ).iloc[0]

        original_rows = result_df[result_df['variant'].eq('original')]

        if len(original_rows):
            original_row = original_rows.iloc[0]

            original_g_over_b = (
                float(original_row['shear_modulus'])
                / (float(original_row['bulk_modulus']) + 1e-12)
            )

            best_g_over_b = float(best_row['metaforge_g_over_b'])

            print('original MetaForge p-ratio:', float(original_row['metaforge_p_ratio']))
            print('original bulk:', float(original_row['bulk_modulus']))
            print('original shear:', float(original_row['shear_modulus']))
            print('original G/B:', original_g_over_b)

            print('best valid MetaForge G/B candidate p-ratio:', float(best_row['metaforge_p_ratio']))
            print('best bulk:', float(best_row['bulk_modulus']))
            print('best shear:', float(best_row['shear_modulus']))
            print('best G/B:', best_g_over_b)

            print('delta p-ratio:', float(best_row['metaforge_p_ratio'] - original_row['metaforge_p_ratio']))
            print('relative bulk change:', float(
                (best_row['bulk_modulus'] - original_row['bulk_modulus'])
                / (abs(original_row['bulk_modulus']) + 1e-12)
            ))
            print('relative shear change:', float(
                (best_row['shear_modulus'] - original_row['shear_modulus'])
                / (abs(original_row['shear_modulus']) + 1e-12)
            ))
            print('relative G/B change:', float(
                (best_g_over_b - original_g_over_b)
                / (abs(original_g_over_b) + 1e-12)
            ))

        else:
            print('best valid MetaForge G/B candidate p-ratio:', float(best_row['metaforge_p_ratio']))

        print(
            best_row[
                [
                    'variant',
                    'reversible_mcmc_score',
                    'linear_ratio_g_over_b',
                    'linear_log_g_over_b',
                    'linear_bulk_like',
                    'linear_shear_like',
                    'near_zero_frac',
                    'min_active_degree',
                    'low_degree_count',
                    'active_connected',
                    'metaforge_p_ratio',
                    'metaforge_g_over_b',
                    'metaforge_log_g_over_b',
                    'bulk_modulus',
                    'shear_modulus',
                    'mechanically_valid',
                ]
            ].to_string()
        )

        display(valid.sort_values(
            ['metaforge_g_over_b', 'metaforge_p_ratio'],
            ascending=[False, True],
        ).head(10))

In [ ]:
# Render optimized networks with graph_utils.
# The MCMC keeps topology fixed, so the useful visual signal is edge stiffness change.

import matplotlib as mpl
import networkx as nx


def pyg_to_stiffness_nx(graph):
    G = nx.Graph()
    xy = graph.x[:, :2].detach().cpu().numpy()
    for node_idx, pos in enumerate(xy):
        G.add_node(int(node_idx), pos=(float(pos[0]), float(pos[1])))

    box = getattr(graph, 'box', None)
    if box is not None:
        if hasattr(box, 'x') and hasattr(box, 'y'):
            G.graph['box'] = {'x': float(box.x), 'y': float(box.y)}
        elif all(hasattr(box, attr) for attr in ('x1', 'x2', 'y1', 'y2')):
            G.graph['box'] = {'x': float(box.x2 - box.x1), 'y': float(box.y2 - box.y1)}

    edge_index = graph.edge_index.detach().cpu().numpy()
    edge_attr = graph.edge_attr.detach().cpu().numpy()
    for edge_col in range(edge_index.shape[1]):
        i, j = int(edge_index[0, edge_col]), int(edge_index[1, edge_col])
        key = tuple(sorted((i, j)))
        stiffness = float(edge_attr[edge_col, -1])
        if not G.has_edge(*key):
            G.add_edge(*key, stiffness=stiffness)
    return G


def pick_render_variants(result_df, candidate_df, values_by_variant, max_extra=3):
    variants = ['original']

    if isinstance(result_df, pd.DataFrame) and not result_df.empty:
        if 'mechanically_valid' in result_df.columns:
            valid_mask = result_df['mechanically_valid'].fillna(False).astype(bool)
        else:
            valid_mask = pd.Series(True, index=result_df.index)
        valid = result_df[valid_mask].copy()
        if not valid.empty and 'metaforge_g_over_b' in valid.columns:
            ranked = valid.sort_values(['metaforge_g_over_b', 'metaforge_p_ratio'], ascending=[False, True])
            variants.extend(ranked['variant'].astype(str).head(max_extra).tolist())

    if len(variants) == 1 and isinstance(candidate_df, pd.DataFrame) and not candidate_df.empty:
        score_cols = [col for col in ['lammps_accept_score', 'reversible_mcmc_score', 'linear_log_g_over_b'] if col in candidate_df.columns]
        if score_cols:
            ranked = candidate_df.sort_values(score_cols[0], ascending=False)
            variants.extend(ranked['variant'].astype(str).head(max_extra).tolist())

    if 'rev_mcmc_best_score' in values_by_variant:
        variants.append('rev_mcmc_best_score')

    out = []
    seen = set()
    for variant in variants:
        if variant in values_by_variant and variant not in seen:
            out.append(variant)
            seen.add(variant)
    return out[: max_extra + 1]


required_globals = ['start_graph', 'keys', 'values_by_variant', 'update_graph_stiffness']
missing_globals = [name for name in required_globals if name not in globals()]
if missing_globals:
    print('Run the reversible MCMC cell first. Missing:', missing_globals)
else:
    render_variants = pick_render_variants(
        globals().get('result_df', pd.DataFrame()),
        globals().get('candidate_df', pd.DataFrame()),
        values_by_variant,
        max_extra=3,
    )
    if not render_variants:
        print('No optimized variants found to render.')
    else:
        base_values = np.asarray(values_by_variant['original'], dtype=float)
        render_graphs = {
            variant: pyg_to_stiffness_nx(update_graph_stiffness(start_graph, keys, values_by_variant[variant]))
            for variant in render_variants
        }

        ratios = []
        for variant in render_variants:
            if variant == 'original':
                continue
            values = np.asarray(values_by_variant[variant], dtype=float)
            ratios.extend(np.log10((values + 1e-12) / (base_values + 1e-12)).tolist())
        if len(ratios):
            max_abs = max(0.25, float(np.nanmax(np.abs(ratios))))
        else:
            max_abs = 1.0
        norm = mpl.colors.TwoSlopeNorm(vmin=-max_abs, vcenter=0.0, vmax=max_abs)
        cmap = mpl.cm.get_cmap('coolwarm')

        ncols = len(render_variants)
        fig, axes = plt.subplots(1, ncols, figsize=(4.2 * ncols, 4.2), constrained_layout=True)
        axes = np.atleast_1d(axes)
        render_rows = []

        for ax, variant in zip(axes, render_variants):
            graph_nx = render_graphs[variant]
            draw_graph(graph_nx, periodic_edges=False, node_color='#E6E8EB', node_size=18, ax=ax, show=False)

            clean_graph = remove_periodic_edges(graph_nx)
            pos = nx.get_node_attributes(clean_graph, 'pos')
            values = np.asarray(values_by_variant[variant], dtype=float)
            value_by_key = {tuple(sorted(key)): float(value) for key, value in zip(keys, values)}
            log_ratio_by_key = {
                tuple(sorted(key)): float(np.log10((value + 1e-12) / (base + 1e-12)))
                for key, value, base in zip(keys, values, base_values)
            }
            edgelist = [tuple(sorted(edge)) for edge in clean_graph.edges]
            edge_log_ratios = [log_ratio_by_key.get(edge, 0.0) for edge in edgelist]
            edge_widths = [0.8 + 2.8 * min(abs(v) / max_abs, 1.0) for v in edge_log_ratios]
            edge_colors = [cmap(norm(v)) for v in edge_log_ratios]
            if variant == 'original':
                edge_colors = ['#5A6573' for _ in edgelist]
                edge_widths = [1.2 for _ in edgelist]

            nx.draw_networkx_edges(
                clean_graph,
                pos=pos,
                edgelist=edgelist,
                edge_color=edge_colors,
                width=edge_widths,
                alpha=0.92,
                ax=ax,
            )

            title = variant
            if isinstance(globals().get('result_df', None), pd.DataFrame) and not result_df.empty:
                rows = result_df[result_df['variant'].astype(str).eq(str(variant))]
                if len(rows):
                    row = rows.iloc[0]
                    pr = row.get('metaforge_p_ratio', np.nan)
                    gb = row.get('metaforge_g_over_b', np.nan)
                    title = f'{variant}\np={pr:.3g}, G/B={gb:.3g}'
                    render_rows.append({
                        'variant': variant,
                        'metaforge_p_ratio': pr,
                        'metaforge_g_over_b': gb,
                        'bulk_modulus': row.get('bulk_modulus', np.nan),
                        'shear_modulus': row.get('shear_modulus', np.nan),
                        'mechanically_valid': row.get('mechanically_valid', np.nan),
                        'changed_edges_gt_2x': int(np.sum(np.abs(np.log2((values + 1e-12) / (base_values + 1e-12))) > 1.0)),
                        'near_zero_edges': int(np.sum(values <= cfg.get('near_zero_stiffness', 1e-7) * 1.01)),
                    })
            ax.set_title(title, fontsize=10)

        sm = mpl.cm.ScalarMappable(norm=norm, cmap=cmap)
        sm.set_array([])
        cbar = fig.colorbar(sm, ax=axes, fraction=0.025, pad=0.02)
        cbar.set_label('log10(stiffness / original)')

        render_dir = output_dir / 'rendered_networks'
        render_dir.mkdir(parents=True, exist_ok=True)
        render_path = render_dir / f'rev_response_mcmc_rendered_networks_sim_{start_sim_idx:03d}.png'
        fig.savefig(render_path, dpi=220)
        plt.show()
        print('saved rendered networks:', render_path)

        if render_rows:
            display(pd.DataFrame(render_rows).round(6))


In [ ]:
# Single-network edge stiffness distribution.

if 'data' not in globals():
    print('Run the data-loading cell first.')
else:
    if 'start_sim_idx' in globals():
        stiffness_sim_idx = int(start_sim_idx)
    elif cfg.get('single_sim_idx') is not None:
        stiffness_sim_idx = int(cfg['single_sim_idx'])
    else:
        stiffness_sim_idx = 0

    stiffness_graph = data[stiffness_sim_idx][0]
    stiffness_keys, stiffness_values = unique_edge_stiffness(stiffness_graph)
    stiffness_values = np.asarray(stiffness_values, dtype=float)

    print('sim_idx:', stiffness_sim_idx)
    print('unique edges:', len(stiffness_values))
    print(pd.Series(stiffness_values, name='edge_stiffness').describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).to_string())

    fig, ax = plt.subplots(figsize=(5.8, 4.0), constrained_layout=True)
    positive = stiffness_values[stiffness_values > 0]
    if len(positive):
        bins = np.logspace(np.log10(max(float(positive.min()), 1e-8)), np.log10(float(positive.max())), 40)
        ax.hist(positive, bins=bins, color=PAPER_COLORS['blue'], alpha=0.78, edgecolor='white', linewidth=0.5)
        ax.set_xscale('log')
    else:
        ax.hist(stiffness_values, bins=40, color=PAPER_COLORS['blue'], alpha=0.78, edgecolor='white', linewidth=0.5)
    ax.set_xlabel('edge stiffness')
    ax.set_ylabel('edge count')
    ax.set_title(f'sim {stiffness_sim_idx}: edge stiffness distribution')
    style_axes(ax)

    single_stiffness_plot_path = output_dir / f'single_network_{stiffness_sim_idx:03d}_edge_stiffness_distribution.png'
    fig.savefig(single_stiffness_plot_path, dpi=180)
    plt.show()
    print('saved:', single_stiffness_plot_path)


In [ ]:
# Edge stiffness distributions for optimized candidates.

required_globals = ['values_by_variant', 'output_dir', 'start_sim_idx']
missing_globals = [name for name in required_globals if name not in globals()]
if missing_globals:
    print('Run the reversible MCMC / final validation cell first. Missing:', missing_globals)
else:
    def pick_stiffness_distribution_variants(result_df, values_by_variant, max_extra=5):
        variants = ['original']
        if isinstance(result_df, pd.DataFrame) and not result_df.empty:
            valid = result_df[result_df.get('mechanically_valid', False).fillna(False)].copy()
            if not valid.empty and {'metaforge_g_over_b', 'metaforge_p_ratio'}.issubset(valid.columns):
                ranked = valid.sort_values(['metaforge_g_over_b', 'metaforge_p_ratio'], ascending=[False, True])
                variants.extend(ranked['variant'].astype(str).head(max_extra).tolist())
            elif 'metaforge_p_ratio' in result_df.columns:
                ranked = result_df.sort_values('metaforge_p_ratio', ascending=True)
                variants.extend(ranked['variant'].astype(str).head(max_extra).tolist())
        if 'rev_mcmc_best_score' in values_by_variant:
            variants.append('rev_mcmc_best_score')

        out = []
        seen = set()
        for variant in variants:
            if variant in values_by_variant and variant not in seen:
                out.append(variant)
                seen.add(variant)
        return out


    stiffness_variants = pick_stiffness_distribution_variants(
        globals().get('result_df', pd.DataFrame()),
        values_by_variant,
        max_extra=5,
    )

    if not stiffness_variants:
        print('No stiffness variants found to plot.')
    else:
        base_values = np.asarray(values_by_variant['original'], dtype=float)
        rows = []
        plot_values = []
        for variant in stiffness_variants:
            values = np.asarray(values_by_variant[variant], dtype=float)
            plot_values.append(values)
            log_ratio = np.log10((values + 1e-12) / (base_values + 1e-12)) if len(values) == len(base_values) else np.full_like(values, np.nan)
            row = {
                'variant': variant,
                'edges': int(len(values)),
                'stiffness_min': float(np.min(values)),
                'stiffness_p01': float(np.quantile(values, 0.01)),
                'stiffness_p05': float(np.quantile(values, 0.05)),
                'stiffness_median': float(np.median(values)),
                'stiffness_mean': float(np.mean(values)),
                'stiffness_p95': float(np.quantile(values, 0.95)),
                'stiffness_max': float(np.max(values)),
                'near_zero_edges': int(np.sum(values <= cfg.get('near_zero_stiffness', 1e-7) * 1.01)),
                'near_zero_frac': float(np.mean(values <= cfg.get('near_zero_stiffness', 1e-7) * 1.01)),
                'weakened_gt_10x': int(np.sum(log_ratio < -1.0)) if np.isfinite(log_ratio).any() else np.nan,
                'strengthened_gt_10x': int(np.sum(log_ratio > 1.0)) if np.isfinite(log_ratio).any() else np.nan,
            }
            if isinstance(globals().get('result_df', None), pd.DataFrame) and not result_df.empty:
                match = result_df[result_df['variant'].astype(str).eq(str(variant))]
                if len(match):
                    row.update({
                        'metaforge_p_ratio': match.iloc[0].get('metaforge_p_ratio', np.nan),
                        'metaforge_g_over_b': match.iloc[0].get('metaforge_g_over_b', np.nan),
                        'bulk_modulus': match.iloc[0].get('bulk_modulus', np.nan),
                        'shear_modulus': match.iloc[0].get('shear_modulus', np.nan),
                    })
            rows.append(row)

        stiffness_distribution_df = pd.DataFrame(rows)
        stiffness_distribution_path = output_dir / f'rev_response_mcmc_stiffness_distribution_sim_{start_sim_idx:03d}.csv'
        stiffness_distribution_df.to_csv(stiffness_distribution_path, index=False)
        display(stiffness_distribution_df.round(6))

        all_positive = np.concatenate([v[v > 0] for v in plot_values if np.any(v > 0)])
        lo = max(float(np.min(all_positive)), 1e-8)
        hi = max(float(np.max(all_positive)), lo * 10)
        bins = np.logspace(np.log10(lo), np.log10(hi), 45)

        fig, axes = plt.subplots(1, 3, figsize=(15.0, 4.2), constrained_layout=True)

        ax = axes[0]
        for variant, values in zip(stiffness_variants, plot_values):
            ax.hist(values, bins=bins, histtype='step', lw=1.6, density=True, label=variant)
        ax.set_xscale('log')
        ax.set_xlabel('edge stiffness')
        ax.set_ylabel('density')
        ax.set_title('Stiffness distribution')
        ax.legend(frameon=False, fontsize=7)
        style_axes(ax)

        ax = axes[1]
        for variant, values in zip(stiffness_variants, plot_values):
            sorted_values = np.sort(values)
            q = np.linspace(0, 1, len(sorted_values), endpoint=True)
            ax.plot(q, sorted_values, lw=1.6, label=variant)
        ax.set_yscale('log')
        ax.set_xlabel('edge quantile')
        ax.set_ylabel('edge stiffness')
        ax.set_title('Sorted stiffness curves')
        style_axes(ax)

        ax = axes[2]
        ratio_bins = np.linspace(-8, 3, 56)
        for variant, values in zip(stiffness_variants, plot_values):
            if variant == 'original' or len(values) != len(base_values):
                continue
            log_ratio = np.log10((values + 1e-12) / (base_values + 1e-12))
            ax.hist(log_ratio, bins=ratio_bins, histtype='step', lw=1.6, density=True, label=variant)
        ax.axvline(0, color='#333333', lw=1, ls='--')
        ax.axvline(-1, color=PAPER_COLORS['red'], lw=1, ls=':', label='10x weaker')
        ax.axvline(1, color=PAPER_COLORS['green'], lw=1, ls=':', label='10x stronger')
        ax.set_xlabel('log10(stiffness / original)')
        ax.set_ylabel('density')
        ax.set_title('Change from original')
        ax.legend(frameon=False, fontsize=7)
        style_axes(ax)

        stiffness_plot_path = output_dir / f'rev_response_mcmc_stiffness_distribution_sim_{start_sim_idx:03d}.png'
        fig.savefig(stiffness_plot_path, dpi=180)
        plt.show()
        print('saved stiffness distribution table:', stiffness_distribution_path)
        print('saved stiffness distribution plot:', stiffness_plot_path)


In [ ]:
# Load Reid trajectories.

data = torch.load(cfg['dataset_path'], map_location='cpu', weights_only=False)
print('trajectories:', len(data))
print('frames per first trajectory:', len(data[0]))
print('nodes:', data[0][0].x.shape[0], 'directed edges:', data[0][0].edge_index.shape[1])


In [ ]:
# Helpers.

def unique_edge_stiffness(graph):
    edge_index = graph.edge_index.detach().cpu().numpy()
    edge_attr = graph.edge_attr.detach().cpu().numpy()
    pair_to_stiffness = {}
    for e in range(edge_index.shape[1]):
        i, j = int(edge_index[0, e]), int(edge_index[1, e])
        key = tuple(sorted((i, j)))
        if key not in pair_to_stiffness:
            pair_to_stiffness[key] = float(edge_attr[e, -1])
    keys = list(pair_to_stiffness.keys())
    values = np.asarray([pair_to_stiffness[k] for k in keys], dtype=float)
    return keys, values


def stiffness_descriptors(values):
    values = np.asarray(values, dtype=float)
    return {
        'stiffness_mean': float(values.mean()),
        'stiffness_std': float(values.std()),
        'stiffness_cv': float(values.std() / (values.mean() + 1e-12)),
        'stiffness_min': float(values.min()),
        'stiffness_max': float(values.max()),
        'soft_frac_lt_0p1': float(np.mean(values < 0.10)),
        'soft_frac_lt_0p2': float(np.mean(values < 0.20)),
    }


def oriented_stiffness_descriptors(values, edge_geometry_df):
    values = np.asarray(values, dtype=float)
    desc = dict(stiffness_descriptors(values))
    for orientation in ('horizontal', 'vertical', 'diagonal'):
        mask = edge_geometry_df['orientation'].eq(orientation).to_numpy()
        subset = values[mask]
        desc[f'{orientation}_stiffness_mean'] = float(subset.mean()) if len(subset) else float('nan')
        desc[f'{orientation}_stiffness_std'] = float(subset.std()) if len(subset) else float('nan')
    return desc


def p_ratio_at_frame(sim, frame_idx):
    side_idx = directional_side_indices_from_box(sim[0], quantile=cfg['side_quantile'])
    frame_idx = min(int(frame_idx), len(sim) - 1)
    return float(calc_p_ratio_rollout_sides(sim, frame_idx, side_idx=side_idx))


def build_summary_from_data():
    rows = []
    for sim_idx, sim in enumerate(data):
        graph0 = sim[0]
        keys, values = unique_edge_stiffness(graph0)
        edge_geometry = unique_edge_geometry(graph0, keys)
        row = {
            'sim_idx': sim_idx,
            'frames': len(sim),
            'nodes': int(sim[0].x.shape[0]),
            'edges_unique': int(len(values)),
            'final_p_ratio': p_ratio_at_frame(sim, cfg['target_frame']),
        }
        row.update(oriented_stiffness_descriptors(values, edge_geometry))
        rows.append(row)
    return pd.DataFrame(rows)


def fit_feature_to_pratio(summary_df):
    x = summary_df[cfg['optimization_feature']].to_numpy(dtype=float)
    y = summary_df['final_p_ratio'].to_numpy(dtype=float)
    slope, intercept = np.polyfit(x, y, deg=1)
    pred = slope * x + intercept
    corr = np.corrcoef(x, y)[0, 1]
    return float(slope), float(intercept), float(corr), pred


def unique_edge_geometry(graph, keys):
    xy = graph.x[:, :2].detach().cpu().numpy()
    x1, x2 = float(graph.box.x1), float(graph.box.x2)
    y1, y2 = float(graph.box.y1), float(graph.box.y2)
    cx, cy = 0.5 * (x1 + x2), 0.5 * (y1 + y2)
    sx, sy = max(abs(x2 - x1), 1e-12), max(abs(y2 - y1), 1e-12)
    rows = []
    for edge_idx, (i, j) in enumerate(keys):
        pi, pj = xy[int(i)], xy[int(j)]
        midpoint = 0.5 * (pi + pj)
        vec = pj - pi
        angle = abs(np.arctan2(vec[1], vec[0]))
        angle = min(angle, np.pi - angle)
        if angle < np.deg2rad(25):
            orientation = 'horizontal'
        elif angle > np.deg2rad(65):
            orientation = 'vertical'
        else:
            orientation = 'diagonal'
        x_norm = (midpoint[0] - cx) / (0.5 * sx)
        y_norm = (midpoint[1] - cy) / (0.5 * sy)
        r_norm = float(np.sqrt(x_norm * x_norm + y_norm * y_norm))
        if r_norm < 0.35:
            region = 'center'
        elif r_norm > 0.75:
            region = 'boundary'
        else:
            region = 'middle'
        rows.append({
            'edge_idx': edge_idx,
            'orientation': orientation,
            'region': region,
            'r_norm': r_norm,
            'angle_deg': float(np.rad2deg(angle)),
            'mid_x_norm': float(x_norm),
            'mid_y_norm': float(y_norm),
        })
    return pd.DataFrame(rows)


def target_edge_indices(edge_geometry_df, target):
    target = str(target)
    if target == 'all':
        mask = np.ones(len(edge_geometry_df), dtype=bool)
    elif target in {'horizontal', 'vertical', 'diagonal'}:
        mask = edge_geometry_df['orientation'].eq(target).to_numpy()
    elif target in {'non_diagonal', 'axial'}:
        mask = edge_geometry_df['orientation'].isin(['horizontal', 'vertical']).to_numpy()
    elif target == 'all':
        mask = np.ones(len(edge_geometry_df), dtype=bool)
    elif '_' in target:
        orientation, region = target.split('_', 1)
        mask = (edge_geometry_df['orientation'].eq(orientation) & edge_geometry_df['region'].eq(region)).to_numpy()
    else:
        raise ValueError(f'Unknown candidate edge target: {target}')
    return edge_geometry_df.loc[mask, 'edge_idx'].to_numpy(dtype=int)


def soften_target_edges_to_near_zero(values, candidate_idx, frac, *, mode='weakest'):
    values = np.asarray(values, dtype=float)
    candidate_idx = np.asarray(candidate_idx, dtype=int)
    out = values.copy()
    if len(candidate_idx) == 0:
        return out, candidate_idx
    n_soft = max(1, int(round(float(frac) * len(candidate_idx))))
    n_soft = min(n_soft, len(candidate_idx))
    candidate_values = values[candidate_idx]
    if mode == 'weakest':
        selected = candidate_idx[np.argsort(candidate_values)[:n_soft]]
    elif mode == 'strongest':
        selected = candidate_idx[np.argsort(candidate_values)[-n_soft:]]
    elif mode == 'random':
        selected = rng.choice(candidate_idx, size=n_soft, replace=False)
    else:
        raise ValueError(f'Unknown soften_selection: {mode}')
    out[selected] = float(cfg['near_zero_stiffness'])
    return np.clip(out, cfg['min_stiffness'], cfg['max_stiffness']), selected


def update_graph_stiffness(graph, keys, new_values):
    out = copy.deepcopy(graph)
    value_by_key = {key: float(value) for key, value in zip(keys, new_values)}
    edge_index = out.edge_index.detach().cpu().numpy()
    edge_attr = out.edge_attr.clone().detach().cpu()
    for e in range(edge_index.shape[1]):
        i, j = int(edge_index[0, e]), int(edge_index[1, e])
        key = tuple(sorted((i, j)))
        edge_attr[e, -1] = value_by_key[key]
    out.edge_attr = edge_attr
    return out


def choose_candidate_scale(candidate_df, slope):
    if cfg['optimization_direction'] == 'force_lower_p_ratio':
        return candidate_df.sort_values('predicted_p_ratio_from_static_fit', ascending=True).iloc[0]
    if cfg['optimization_direction'] in ('force_higher_p_ratio', 'fit_high_p_ratio'):
        return candidate_df.sort_values('predicted_p_ratio_from_static_fit', ascending=False).iloc[0]
    if cfg['optimization_direction'] == 'force_higher_std':
        return candidate_df.sort_values('stiffness_std', ascending=False).iloc[0]
    if cfg['optimization_direction'] == 'force_lower_std':
        return candidate_df.sort_values('stiffness_std', ascending=True).iloc[0]
    raise ValueError(f"Unknown optimization_direction: {cfg['optimization_direction']}")


def run_metaforge_elastic(graph, label):
    return run_elastic_simulation(
        graph,
        parent_dir=run_dir,
        label=label,
        config=metaforge_elastic_config(),
        box=graph.box,
    )


In [ ]:
# Build descriptor table and fit the static p-ratio signal.

source_path = Path(cfg['source_08_descriptors'])
if source_path.exists():
    summary_df = pd.read_csv(source_path)
    if 'target_frame_idx' in summary_df.columns:
        summary_df = summary_df.rename(columns={'target_frame_idx': 'source_target_frame_idx'})
else:
    summary_df = build_summary_from_data()

static_features = list(cfg.get('static_prediction_features', [cfg['optimization_feature']]))
needed = {'sim_idx', 'final_p_ratio', *static_features}
missing_from_source = needed.difference(summary_df.columns)
if missing_from_source:
    summary_from_data = build_summary_from_data()
    merge_cols = ['sim_idx'] + [col for col in missing_from_source if col in summary_from_data.columns]
    summary_df = summary_df.merge(summary_from_data[merge_cols], on='sim_idx', how='left')

missing = needed.difference(summary_df.columns)
if missing:
    raise ValueError(f'Missing columns in descriptor table: {sorted(missing)}')

slope, intercept, corr, fit_pred_single = fit_feature_to_pratio(summary_df)

x_static = summary_df[static_features].to_numpy(dtype=float)
y_static = summary_df['final_p_ratio'].to_numpy(dtype=float)
x_mean = x_static.mean(axis=0, keepdims=True)
x_std = x_static.std(axis=0, keepdims=True) + 1e-12
x_design = np.c_[np.ones(len(x_static)), (x_static - x_mean) / x_std]
static_coef = np.linalg.lstsq(x_design, y_static, rcond=None)[0]
fit_pred = x_design @ static_coef
summary_df['predicted_p_ratio_from_static_fit'] = fit_pred
summary_df['predicted_p_ratio_from_std_only_fit'] = fit_pred_single
summary_df.to_csv(output_dir / 'source_static_descriptors_with_fit.csv', index=False)


def predict_p_ratio_from_descriptors(desc):
    x = np.asarray([float(desc[name]) for name in static_features], dtype=float).reshape(1, -1)
    x_norm = (x - x_mean) / x_std
    return float(np.r_[1.0, x_norm.ravel()] @ static_coef)

print(f"{cfg['optimization_feature']} -> p_ratio: Pearson r={corr:.4f}, R2={corr**2:.4f}")
for feature in static_features:
    feature_corr = np.corrcoef(summary_df[feature].to_numpy(float), y_static)[0, 1]
    print(f"{feature} -> p_ratio: Pearson r={feature_corr:.4f}, R2={feature_corr**2:.4f}")
print('static prediction features:', static_features)
print('standardized coefficients:', {'intercept': static_coef[0], **{name: value for name, value in zip(static_features, static_coef[1:])}})
print('positive coefficient means increasing the standardized feature raises predicted p-ratio; negative means increasing it lowers predicted p-ratio.')

display_cols = ['sim_idx', *static_features, 'final_p_ratio', 'predicted_p_ratio_from_static_fit']
display(summary_df.sort_values(cfg['select_by'], ascending=False).head(10)[display_cols].round(4))

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.0), constrained_layout=True)
scatter_specs = [
    ('stiffness_std', 'stiffness std', axes[0]),
    ('stiffness_mean', 'stiffness mean', axes[1]),
]
for feature, xlabel, ax in scatter_specs:
    ax.scatter(summary_df[feature], summary_df['final_p_ratio'], s=28, alpha=0.72, edgecolor='white', linewidth=0.4)
    ax.set_xlabel(xlabel)
    ax.set_ylabel('final p-ratio')
    ax.set_title(f'{xlabel} vs p-ratio')
    style_axes(ax)
fig.savefig(output_dir / 'source_stiffness_scatter.png', dpi=180)
plt.show()


In [ ]:
# Choose one Reid network and build candidate stiffness interventions.

def resolve_single_sim_idx():
    if cfg['single_sim_idx'] is not None:
        return int(cfg['single_sim_idx'])
    return int(summary_df.sort_values(cfg['select_by'], ascending=False).iloc[0]['sim_idx'])


def _safe_name(value):
    return str(value).replace('.', 'p').replace('-', 'm').replace('+', 'p')


def weighted_random_edge_indices(edge_geometry_df, n_samples, diagonal_weight=4.0):
    weights = np.where(edge_geometry_df['orientation'].eq('diagonal').to_numpy(), float(diagonal_weight), 1.0)
    weights = weights / weights.sum()
    n_samples = min(int(n_samples), len(edge_geometry_df))
    return rng.choice(edge_geometry_df['edge_idx'].to_numpy(dtype=int), size=n_samples, replace=False, p=weights)


single_sim_idx = resolve_single_sim_idx()
single_sim = data[single_sim_idx]
single_graph0 = single_sim[0]
single_keys, single_values = unique_edge_stiffness(single_graph0)
single_edge_geometry = unique_edge_geometry(single_graph0, single_keys)
print('single sim_idx:', single_sim_idx)
print('original rollout p-ratio t100:', p_ratio_at_frame(single_sim, cfg['target_frame']))
print('static prediction features:', static_features)

support_factor_grid = [float(x) for x in cfg.get('support_factor_grid', [1.25, 1.5, 2.0, 3.0, 4.0])]
sampled_edge_idxs = weighted_random_edge_indices(
    single_edge_geometry,
    cfg.get('random_candidate_count', 6),
    cfg.get('diagonal_sample_weight', 4.0),
)

candidate_rows = []
candidate_graphs = {}
original_desc = stiffness_descriptors(single_values)
original_pred = predict_p_ratio_from_descriptors(original_desc)
candidate_rows.append({
    'sim_idx': single_sim_idx,
    'variant': 'original',
    'candidate_kind': 'original',
    'selected_edge_idx': -1,
    'selected_edge_orientation': 'none',
    'selected_edge_sample_weight': 0.0,
    'soften_target': 'none',
    'soften_frac': 0.0,
    'support_target': 'none',
    'support_factor': 1.0,
    'support_factor_resolved': 1.0,
    'support_scale_label': 'none',
    'ops': 'original',
    'predicted_p_ratio_from_static_fit': original_pred,
    **original_desc,
})
candidate_graphs['original'] = single_graph0

for sample_i in range(len(sampled_edge_idxs)):
    softened_edge_idxs = sampled_edge_idxs[: sample_i + 1]
    softened_rows = single_edge_geometry.loc[single_edge_geometry['edge_idx'].isin(softened_edge_idxs)]
    selected_orientation = ','.join(softened_rows['orientation'].astype(str).tolist())
    sample_weight = float(np.mean(np.where(softened_rows['orientation'].eq('diagonal'), float(cfg.get('diagonal_sample_weight', 4.0)), 1.0))) if len(softened_rows) else 1.0
    for support_factor in support_factor_grid:
        new_values = np.asarray(single_values, dtype=float).copy()
        new_values[softened_edge_idxs.astype(int)] = float(cfg['near_zero_stiffness'])
        new_values = np.clip(new_values * float(support_factor), cfg['min_stiffness'], cfg['max_stiffness'])
        desc = stiffness_descriptors(new_values)
        pred = predict_p_ratio_from_descriptors(desc)
        variant = f"rand_{sample_i+1:02d}edges__scale_all_{_safe_name(support_factor)}x"
        if variant in candidate_graphs:
            continue
        op_records = [
            {
                'action': 'soften_to_near_zero',
                'target': 'random_edges',
                'selected_edge_idxs': [int(x) for x in softened_edge_idxs],
                'selected_orientation': selected_orientation,
                'sample_weight': sample_weight,
                'frac': float((sample_i + 1) / max(len(single_values), 1)),
                'factor': np.nan,
                'factor_resolved': np.nan,
                'static_predicted_p_ratio': np.nan,
                'selected_count': int(len(softened_edge_idxs)),
                'candidate_count': int(len(single_values)),
            },
            {
                'action': 'multiply',
                'target': 'all',
                'frac': 1.0,
                'factor': float(support_factor),
                'factor_resolved': float(support_factor),
                'static_predicted_p_ratio': np.nan,
                'selected_count': int(len(single_values)),
                'candidate_count': int(len(single_values)),
            },
        ]
        candidate_rows.append({
            'sim_idx': single_sim_idx,
            'variant': variant,
            'candidate_kind': 'random_edges_progression',
            'softened_edge_count': int(len(softened_edge_idxs)),
            'softened_edge_idxs': json.dumps([int(x) for x in softened_edge_idxs]),
            'selected_edge_orientation': selected_orientation,
            'selected_edge_sample_weight': sample_weight,
            'soften_target': 'random_edges',
            'soften_frac': float((sample_i + 1) / max(len(single_values), 1)),
            'support_target': 'all',
            'support_factor': float(support_factor),
            'support_factor_resolved': float(support_factor),
            'support_scale_label': f'scale all {support_factor:.3g}x',
            'ops': json.dumps(op_records),
            'predicted_p_ratio_from_static_fit': pred,
            **desc,
        })
        candidate_graphs[variant] = update_graph_stiffness(single_graph0, single_keys, new_values)

candidate_df = pd.DataFrame(candidate_rows)
if 'support_scale_label' not in candidate_df.columns:
    candidate_df['support_scale_label'] = 'none'
fallback_support_label = pd.Series(
    np.where(candidate_df['support_target'].eq('none'), 'none', 'scale ' + candidate_df['support_target'].astype(str) + ' ' + candidate_df['support_factor_resolved'].astype(str) + 'x'),
    index=candidate_df.index,
)
candidate_df['support_scale_label'] = candidate_df['support_scale_label'].fillna(fallback_support_label)
candidate_df.to_csv(output_dir / f'single_network_{single_sim_idx:03d}_candidates.csv', index=False)
print('candidate count excluding original:', len(candidate_df) - 1)
print(candidate_df.sort_values('predicted_p_ratio_from_static_fit')[['variant', 'softened_edge_count', 'selected_edge_orientation', 'support_scale_label', 'stiffness_mean', 'stiffness_std', 'predicted_p_ratio_from_static_fit']].round(4).to_string(index=False))


In [ ]:
# Plot static-fit candidate ranking before real simulation.

plot_rank = candidate_df[candidate_df['variant'].ne('original')].copy().sort_values('predicted_p_ratio_from_static_fit')
fig, axes = plt.subplots(1, 2, figsize=(12.0, 4.4), constrained_layout=True)

ax = axes[0]
colors = np.where(plot_rank['softened_edge_count'] % 2 == 0, '#4C78A8', '#E45756')
ax.scatter(plot_rank['stiffness_mean'], plot_rank['stiffness_std'], c=colors, s=54, alpha=0.85, edgecolor='white', linewidth=0.5)
ax.set_xlabel('candidate stiffness mean')
ax.set_ylabel('candidate stiffness std')
ax.set_title('Random-edge candidates')
style_axes(ax)

ax = axes[1]
label = plot_rank['softened_edge_count'].astype(str) + ' edges | ' + plot_rank['support_scale_label']
y = np.arange(len(plot_rank))[::-1]
ax.barh(y, plot_rank['predicted_p_ratio_from_static_fit'], color=colors)
ax.axvline(0, color='#333333', lw=1, ls='--')
ax.set_yticks(y)
ax.set_yticklabels(label, fontsize=7)
ax.set_xlabel('static-fit predicted p-ratio')
ax.set_title('Static predicted p-ratio')
style_axes(ax)

fig.savefig(output_dir / f'single_network_{single_sim_idx:03d}_candidate_static_ranking.png', dpi=180)
plt.show()


In [ ]:
# Run MetaForge LAMMPS scan for the single-network candidates.

if cfg['run_lammps'] and shutil.which(cfg['lammps_cmd']) is None:
    raise FileNotFoundError(f"LAMMPS command not found: {cfg['lammps_cmd']}")

lammps_rows = []
if cfg['run_lammps']:
    for _, row in candidate_df.iterrows():
        variant = str(row['variant'])
        graph = candidate_graphs[variant]
        label = f'single_{single_sim_idx:03d}_{variant}'[:180]
        print('running', label)
        result = run_metaforge_elastic(graph, label)
        out = {'sim_idx': single_sim_idx, 'variant': variant, **result}
        out['mechanically_valid'] = (
            np.isfinite(out['metaforge_p_ratio'])
            and out['bulk_modulus'] > float(cfg['valid_bulk_min'])
            and out['shear_modulus'] > float(cfg['valid_shear_min'])
        )
        lammps_rows.append(out)
        print(f"  p-ratio={out['metaforge_p_ratio']:.6g}, bulk={out['bulk_modulus']:.6g}, shear={out['shear_modulus']:.6g}, valid={out['mechanically_valid']}")
else:
    print('cfg[run_lammps] is False, skipping MetaForge scan.')

lammps_df = pd.DataFrame(lammps_rows)
if not lammps_df.empty:
    result_df = candidate_df.merge(lammps_df, on=['sim_idx', 'variant'], how='left')
    if 'support_scale_label' not in result_df.columns:
        result_df['support_scale_label'] = np.where(result_df['support_target'].eq('none'), 'none', 'scale ' + result_df['support_target'].astype(str) + ' ' + result_df['support_factor_resolved'].astype(str) + 'x')
    result_df.to_csv(output_dir / f'single_network_{single_sim_idx:03d}_metaforge_scan.csv', index=False)
    valid = result_df[result_df['mechanically_valid'].fillna(False)].copy()
    if valid.empty:
        print('No mechanically valid candidate found.')
    else:
        best_row = valid.sort_values('metaforge_p_ratio').iloc[0]
        original_row = result_df[result_df['variant'].eq('original')].iloc[0]
        print('original MetaForge p-ratio:', float(original_row['metaforge_p_ratio']))
        print('best valid MetaForge p-ratio:', float(best_row['metaforge_p_ratio']))
        print('delta:', float(best_row['metaforge_p_ratio'] - original_row['metaforge_p_ratio']))
        print(best_row[['variant', 'soften_target', 'soften_frac', 'support_target', 'support_factor', 'metaforge_p_ratio', 'bulk_modulus', 'shear_modulus']].to_string())


In [ ]:
# MCMC optimization toward low-p-ratio Reid-like networks, then MetaForge/LAMMPS validation.

cfg.update({
    'mcmc_target_p_ratio': -0.1,
    'mcmc_steps': 50000,
    'mcmc_burn_in': 200,
    'mcmc_keep_every': 10,
    'mcmc_temperature': 0.35,
    'mcmc_prediction_weight': 4.0,
    'mcmc_distribution_weight': 1.0,
    'mcmc_soften_min_frac': 0.05,
    'mcmc_soften_max_frac': 0.35,
    'mcmc_soften_bias': 'diagonal',
    'mcmc_diagonal_edge_weight': 4.0,
    'mcmc_top_k_to_simulate': 20,
    'mcmc_start_sim_idx': cfg.get('single_sim_idx', None),
})
cfg.update({
    # Probability of choosing the "set edge to near-zero" move.
    # The remaining probability is the "increase edge stiffness by 1.5x" move.
    'mcmc_prob_near_zero_move': 0.1,

    # Move A: almost delete/soften one edge.
    'mcmc_near_zero_value': cfg.get('near_zero_stiffness', 1e-7),

    # Move B: stiffen one edge.
    'mcmc_stiffen_factor': 2,

    # Bias edge selection. Keep diagonal bias for both moves by default.
    'mcmc_edge_selection_bias': 'diagonal',
    'mcmc_diagonal_edge_weight': 2.0,
})

def unique_edge_stiffness(graph):
    edge_index = graph.edge_index.detach().cpu().numpy()
    edge_attr = graph.edge_attr.detach().cpu().numpy()

    pair_to_stiffness = {}
    for e in range(edge_index.shape[1]):
        i, j = int(edge_index[0, e]), int(edge_index[1, e])
        key = tuple(sorted((i, j)))
        if key not in pair_to_stiffness:
            pair_to_stiffness[key] = float(edge_attr[e, -1])

    keys = list(pair_to_stiffness.keys())
    values = np.asarray([pair_to_stiffness[k] for k in keys], dtype=float)
    return keys, values


def update_graph_stiffness(graph, keys, new_values):
    out = copy.deepcopy(graph)
    value_by_key = {key: float(value) for key, value in zip(keys, new_values)}

    edge_index = out.edge_index.detach().cpu().numpy()
    edge_attr = out.edge_attr.clone().detach().cpu()

    for e in range(edge_index.shape[1]):
        i, j = int(edge_index[0, e]), int(edge_index[1, e])
        key = tuple(sorted((i, j)))
        edge_attr[e, -1] = value_by_key[key]

    out.edge_attr = edge_attr
    return out


def unique_edge_geometry(graph, keys):
    xy = graph.x[:, :2].detach().cpu().numpy()

    x1, x2 = float(graph.box.x1), float(graph.box.x2)
    y1, y2 = float(graph.box.y1), float(graph.box.y2)

    cx, cy = 0.5 * (x1 + x2), 0.5 * (y1 + y2)
    sx, sy = max(abs(x2 - x1), 1e-12), max(abs(y2 - y1), 1e-12)

    rows = []
    for edge_idx, (i, j) in enumerate(keys):
        pi, pj = xy[int(i)], xy[int(j)]
        midpoint = 0.5 * (pi + pj)
        vec = pj - pi

        angle = abs(np.arctan2(vec[1], vec[0]))
        angle = min(angle, np.pi - angle)

        if angle < np.deg2rad(25):
            orientation = 'horizontal'
        elif angle > np.deg2rad(65):
            orientation = 'vertical'
        else:
            orientation = 'diagonal'

        x_norm = (midpoint[0] - cx) / (0.5 * sx)
        y_norm = (midpoint[1] - cy) / (0.5 * sy)
        r_norm = float(np.sqrt(x_norm * x_norm + y_norm * y_norm))

        if r_norm < 0.35:
            region = 'center'
        elif r_norm > 0.75:
            region = 'boundary'
        else:
            region = 'middle'

        rows.append({
            'edge_idx': edge_idx,
            'orientation': orientation,
            'region': region,
            'r_norm': r_norm,
            'angle_deg': float(np.rad2deg(angle)),
            'mid_x_norm': float(x_norm),
            'mid_y_norm': float(y_norm),
        })

    return pd.DataFrame(rows)


def stiffness_descriptors(values):
    values = np.asarray(values, dtype=float)
    return {
        'stiffness_mean': float(values.mean()),
        'stiffness_std': float(values.std()),
        'stiffness_cv': float(values.std() / (values.mean() + 1e-12)),
        'stiffness_min': float(values.min()),
        'stiffness_max': float(values.max()),
        'soft_frac_lt_0p1': float(np.mean(values < 0.10)),
        'soft_frac_lt_0p2': float(np.mean(values < 0.20)),
        'soft_frac_lt_0p5': float(np.mean(values < 0.50)),
    }


def oriented_stiffness_descriptors(values, edge_geometry_df):
    values = np.asarray(values, dtype=float)
    desc = dict(stiffness_descriptors(values))

    for orientation in ('horizontal', 'vertical', 'diagonal'):
        mask = edge_geometry_df['orientation'].eq(orientation).to_numpy()
        subset = values[mask]

        desc[f'{orientation}_stiffness_mean'] = float(subset.mean()) if len(subset) else float('nan')
        desc[f'{orientation}_stiffness_std'] = float(subset.std()) if len(subset) else float('nan')
        desc[f'{orientation}_soft_frac_lt_0p2'] = float(np.mean(subset < 0.20)) if len(subset) else float('nan')

    return desc


def p_ratio_at_frame(sim, frame_idx):
    side_idx = directional_side_indices_from_box(sim[0], quantile=cfg['side_quantile'])
    frame_idx = min(int(frame_idx), len(sim) - 1)
    return float(calc_p_ratio_rollout_sides(sim, frame_idx, side_idx=side_idx))


def build_summary_from_data(data):
    rows = []

    for sim_idx, sim in enumerate(data):
        graph0 = sim[0]
        keys, values = unique_edge_stiffness(graph0)
        edge_geometry_df = unique_edge_geometry(graph0, keys)

        row = {
            'sim_idx': sim_idx,
            'frames': len(sim),
            'nodes': int(graph0.x.shape[0]),
            'edges_unique': int(len(values)),
            'final_p_ratio': p_ratio_at_frame(sim, cfg['target_frame']),
        }

        row.update(oriented_stiffness_descriptors(values, edge_geometry_df))
        rows.append(row)

    return pd.DataFrame(rows)


def fit_linear_surrogate(summary_df, feature_cols):
    X = summary_df[feature_cols].to_numpy(dtype=float)
    y = summary_df['final_p_ratio'].to_numpy(dtype=float)

    x_mean = X.mean(axis=0)
    x_std = X.std(axis=0) + 1e-12

    Xz = (X - x_mean) / x_std
    X_aug = np.column_stack([np.ones(len(Xz)), Xz])

    coef = np.linalg.lstsq(X_aug, y, rcond=None)[0]
    pred = X_aug @ coef

    return {
        'coef': coef,
        'x_mean': x_mean,
        'x_std': x_std,
        'feature_cols': feature_cols,
        'train_corr': float(np.corrcoef(pred, y)[0, 1]),
        'train_rmse': float(np.sqrt(np.mean((pred - y) ** 2))),
    }


def predict_p_ratio_from_values(values, edge_geometry_df, surrogate):
    desc = oriented_stiffness_descriptors(values, edge_geometry_df)
    x = np.asarray([desc[c] for c in surrogate['feature_cols']], dtype=float)
    xz = (x - surrogate['x_mean']) / surrogate['x_std']
    x_aug = np.r_[1.0, xz]
    return float(x_aug @ surrogate['coef']), desc


def fit_target_gaussian(summary_df, feature_cols, target_mask):
    X = summary_df.loc[target_mask, feature_cols].to_numpy(dtype=float)

    mu = X.mean(axis=0)
    sigma = X.std(axis=0) + 1e-12
    Z = (X - mu) / sigma

    cov = np.cov(Z, rowvar=False)
    cov = np.atleast_2d(cov)
    cov = cov + 1e-4 * np.eye(cov.shape[0])

    inv_cov = np.linalg.pinv(cov)
    log_det = float(np.linalg.slogdet(cov)[1])

    return {
        'mu': mu,
        'sigma': sigma,
        'inv_cov': inv_cov,
        'log_det': log_det,
        'feature_cols': feature_cols,
        'n_target': int(X.shape[0]),
    }


def target_log_prob_from_desc(desc, target_model):
    x = np.asarray([desc[c] for c in target_model['feature_cols']], dtype=float)
    z = (x - target_model['mu']) / target_model['sigma']
    q = float(z @ target_model['inv_cov'] @ z)
    return -0.5 * (q + target_model['log_det'])


def mcmc_score(values, edge_geometry_df, surrogate, target_model=None):
    pred_p_ratio, desc = predict_p_ratio_from_values(values, edge_geometry_df, surrogate)

    score = -pred_p_ratio

    return float(score), float(pred_p_ratio), desc, np.nan


def edge_selection_weights(values, edge_geometry_df, move_type):
    values = np.asarray(values, dtype=float)
    weights = np.ones(len(values), dtype=float)

    if cfg.get('mcmc_edge_selection_bias', 'diagonal') == 'diagonal':
        diagonal_mask = edge_geometry_df['orientation'].eq('diagonal').to_numpy()
        weights[diagonal_mask] *= float(cfg['mcmc_diagonal_edge_weight'])

    if move_type == 'near_zero':
        movable = values > float(cfg['min_stiffness']) * 10
    elif move_type == 'stiffen':
        movable = values < float(cfg['max_stiffness']) / float(cfg['mcmc_stiffen_factor'])
    else:
        raise ValueError(f'Unknown move_type: {move_type}')

    weights[~movable] = 0.0

    if weights.sum() <= 0:
        return None

    return weights / weights.sum()


def propose_mcmc_edge_move(values, edge_geometry_df, rng):
    values = np.asarray(values, dtype=float)
    proposal = values.copy()

    if rng.random() < float(cfg['mcmc_prob_near_zero_move']):
        move_type = 'near_zero'
    else:
        move_type = 'stiffen'

    weights = edge_selection_weights(values, edge_geometry_df, move_type)

    # If the chosen move is impossible, try the other move.
    if weights is None:
        move_type = 'stiffen' if move_type == 'near_zero' else 'near_zero'
        weights = edge_selection_weights(values, edge_geometry_df, move_type)

    if weights is None:
        return proposal, None, 'none', 1.0, np.nan, np.nan

    edge_idx = int(rng.choice(np.arange(len(values)), p=weights))

    old_value = float(proposal[edge_idx])

    if move_type == 'near_zero':
        new_value = float(cfg['mcmc_near_zero_value'])
    elif move_type == 'stiffen':
        new_value = old_value * float(cfg['mcmc_stiffen_factor'])
    else:
        raise ValueError(f'Unknown move_type: {move_type}')

    new_value = float(np.clip(new_value, float(cfg['min_stiffness']), float(cfg['max_stiffness'])))
    proposal[edge_idx] = new_value

    move_factor = new_value / (old_value + 1e-12)

    return proposal, edge_idx, move_type, move_factor, old_value, new_value
def run_mcmc_for_graph(start_graph, start_sim_idx, surrogate, target_model, rng):
    keys, start_values = unique_edge_stiffness(start_graph)
    edge_geometry_df = unique_edge_geometry(start_graph, keys)

    current_values = start_values.copy()
    current_score, current_pred, current_desc, current_logp = mcmc_score(
        current_values,
        edge_geometry_df,
        surrogate,
        target_model,
    )

    records = []
    kept_graphs = {}

    best_values = current_values.copy()
    best_score = current_score
    best_pred = current_pred

    for step in range(int(cfg['mcmc_steps']) + 1):
        if step > 0:
            proposal_values, changed_edge_idx, move_type, move_factor, old_edge_value, new_edge_value = propose_mcmc_edge_move(
                current_values,
                edge_geometry_df,
                rng,
            )

            proposal_score, proposal_pred, proposal_desc, proposal_logp = mcmc_score(
                proposal_values,
                edge_geometry_df,
                surrogate,
                target_model,
            )

            delta = proposal_score - current_score
            accept_prob = min(1.0, float(np.exp(delta / float(cfg['mcmc_temperature']))))
            accepted = bool(rng.random() < accept_prob)

            if accepted:
                current_values = proposal_values
                current_score = proposal_score
                current_pred = proposal_pred
                current_desc = proposal_desc
                current_logp = proposal_logp

                if current_score > best_score:
                    best_score = current_score
                    best_pred = current_pred
                    best_values = current_values.copy()
        else:
            changed_edge_idx = None
            soften_factor = 1.0
            accept_prob = 1.0
            accepted = True

        keep = (
            step >= int(cfg['mcmc_burn_in'])
            and step % int(cfg['mcmc_keep_every']) == 0
        )

        row = {
            'sim_idx': int(start_sim_idx),
            'step': int(step),
            'accepted': bool(accepted),
            'accept_prob': float(accept_prob),
            'changed_edge_idx': changed_edge_idx,
            'soften_factor': float(soften_factor),
            'mcmc_score': float(current_score),
            'target_distribution_log_prob': float(current_logp),
            'predicted_p_ratio': float(current_pred),
            'best_score_so_far': float(best_score),
            'best_predicted_p_ratio_so_far': float(best_pred),
        }

        row.update({f'current_{k}': v for k, v in current_desc.items()})
        records.append(row)

        if keep:
            variant = f'mcmc_step_{step:05d}'
            kept_graphs[variant] = update_graph_stiffness(start_graph, keys, current_values)

        if step % 100 == 0:
            print(
                f"step={step:5d} score={current_score:9.3f} "
                f"pred_p={current_pred:8.4f} best_pred={best_pred:8.4f} "
                f"accepted={accepted}"
            )

    kept_graphs['mcmc_best_static_score'] = update_graph_stiffness(start_graph, keys, best_values)

    trace_df = pd.DataFrame(records)
    return trace_df, kept_graphs, edge_geometry_df


def run_metaforge_elastic(graph, label):
    return run_elastic_simulation(
        graph,
        parent_dir=run_dir,
        label=label,
        config=metaforge_elastic_config(),
        box=graph.box,
    )


data = torch.load(cfg['dataset_path'], map_location='cpu', weights_only=False)

print('trajectories:', len(data))
print('frames per first trajectory:', len(data[0]))
print('nodes:', data[0][0].x.shape[0], 'directed edges:', data[0][0].edge_index.shape[1])

summary_path = output_dir / 'reid_mcmc_static_summary.csv'

if summary_path.exists():
    summary_df = pd.read_csv(summary_path)
else:
    summary_df = build_summary_from_data(data)
    summary_df.to_csv(summary_path, index=False)

target_mask = summary_df['final_p_ratio'] < float(cfg['mcmc_target_p_ratio'])

print('target networks:', int(target_mask.sum()), '/', len(summary_df))
print(summary_df['final_p_ratio'].describe().to_string())

if target_mask.sum() < 5:
    raise ValueError(
        f"Only {int(target_mask.sum())} networks have p-ratio < {cfg['mcmc_target_p_ratio']}. "
        "Relax cfg['mcmc_target_p_ratio'] or add more data."
    )

feature_cols = [
    'stiffness_mean',
    'stiffness_std',
    'stiffness_cv',
    'stiffness_min',
    'stiffness_max',
    'soft_frac_lt_0p1',
    'soft_frac_lt_0p2',
    'soft_frac_lt_0p5',
    'horizontal_stiffness_mean',
    'horizontal_stiffness_std',
    'horizontal_soft_frac_lt_0p2',
    'vertical_stiffness_mean',
    'vertical_stiffness_std',
    'vertical_soft_frac_lt_0p2',
    'diagonal_stiffness_mean',
    'diagonal_stiffness_std',
    'diagonal_soft_frac_lt_0p2',
]

feature_cols = [c for c in feature_cols if c in summary_df.columns and np.isfinite(summary_df[c]).all()]

surrogate = fit_linear_surrogate(summary_df, feature_cols)
target_model = fit_target_gaussian(summary_df, feature_cols, target_mask)

print('surrogate train corr:', surrogate['train_corr'])
print('surrogate train rmse:', surrogate['train_rmse'])
print('target gaussian n:', target_model['n_target'])

if cfg['mcmc_start_sim_idx'] is None:
    start_sim_idx = int(summary_df.sort_values('final_p_ratio', ascending=False).iloc[0]['sim_idx'])
else:
    start_sim_idx = int(cfg['mcmc_start_sim_idx'])

start_graph = data[start_sim_idx][0]

print('starting sim_idx:', start_sim_idx)
print('starting rollout p-ratio:', float(summary_df.loc[summary_df['sim_idx'].eq(start_sim_idx), 'final_p_ratio'].iloc[0]))

trace_df, candidate_graphs, edge_geometry_df = run_mcmc_for_graph(
    start_graph=start_graph,
    start_sim_idx=start_sim_idx,
    surrogate=surrogate,
    target_model=target_model,
    rng=rng,
)

trace_path = output_dir / f'mcmc_trace_sim_{start_sim_idx:03d}.csv'
trace_df.to_csv(trace_path, index=False)

kept_rows = []

for variant, graph in candidate_graphs.items():
    keys, values = unique_edge_stiffness(graph)
    pred, desc = predict_p_ratio_from_values(values, edge_geometry_df, surrogate)
    score, pred2, desc2, logp = mcmc_score(values, edge_geometry_df, surrogate, target_model)

    row = {
        'sim_idx': start_sim_idx,
        'variant': variant,
        'predicted_p_ratio': float(pred),
        'mcmc_score': float(score),
        'target_distribution_log_prob': float(logp),
    }
    row.update(desc)
    kept_rows.append(row)

candidate_df = pd.DataFrame(kept_rows)
candidate_df = candidate_df.sort_values(
    ['predicted_p_ratio'],
    ascending=[True],
).reset_index(drop=True)

candidate_path = output_dir / f'mcmc_candidates_sim_{start_sim_idx:03d}.csv'
candidate_df.to_csv(candidate_path, index=False)

print('saved trace:', trace_path)
print('saved candidates:', candidate_path)

display(candidate_df.head(20))

if cfg['run_lammps'] and shutil.which(cfg['lammps_cmd']) is None:
    raise FileNotFoundError(f"LAMMPS command not found: {cfg['lammps_cmd']}")

lammps_rows = []

if cfg['run_lammps']:
    top_variants = candidate_df.head(int(cfg['mcmc_top_k_to_simulate']))['variant'].tolist()

    if 'original' not in candidate_graphs:
        candidate_graphs['original'] = start_graph
        top_variants = ['original'] + top_variants

    for variant in top_variants:
        graph = candidate_graphs[variant]
        label = f'mcmc_{start_sim_idx:03d}_{variant}'[:180]

        print('running', label)

        try:
            result = run_metaforge_elastic(graph, label)
            out = {
                'sim_idx': start_sim_idx,
                'variant': variant,
                **result,
            }

            out['mechanically_valid'] = (
                np.isfinite(out['metaforge_p_ratio'])
                and out['bulk_modulus'] > float(cfg['valid_bulk_min'])
                and out['shear_modulus'] > float(cfg['valid_shear_min'])
            )

            print(
                f"  p-ratio={out['metaforge_p_ratio']:.6g}, "
                f"bulk={out['bulk_modulus']:.6g}, "
                f"shear={out['shear_modulus']:.6g}, "
                f"valid={out['mechanically_valid']}"
            )

        except Exception as exc:
            out = {
                'sim_idx': start_sim_idx,
                'variant': variant,
                'metaforge_p_ratio': np.nan,
                'bulk_modulus': np.nan,
                'shear_modulus': np.nan,
                'mechanically_valid': False,
                'error': repr(exc),
            }
            print('  failed:', repr(exc))

        lammps_rows.append(out)

else:
    print('cfg[run_lammps] is False, skipping MetaForge validation.')

lammps_df = pd.DataFrame(lammps_rows)

if not lammps_df.empty:
    result_df = candidate_df.merge(lammps_df, on=['sim_idx', 'variant'], how='right')
    result_path = output_dir / f'mcmc_metaforge_results_sim_{start_sim_idx:03d}.csv'
    result_df.to_csv(result_path, index=False)

    print('saved MetaForge results:', result_path)

    valid = result_df[result_df['mechanically_valid'].fillna(False)].copy()

    if valid.empty:
        print('No mechanically valid MCMC candidate found.')
    else:
        best_row = valid.sort_values('metaforge_p_ratio', ascending=True).iloc[0]

        original_rows = result_df[result_df['variant'].eq('original')]
        if len(original_rows):
            original_row = original_rows.iloc[0]
            print('original MetaForge p-ratio:', float(original_row['metaforge_p_ratio']))
            print('best valid MetaForge p-ratio:', float(best_row['metaforge_p_ratio']))
            print('delta:', float(best_row['metaforge_p_ratio'] - original_row['metaforge_p_ratio']))
        else:
            print('best valid MetaForge p-ratio:', float(best_row['metaforge_p_ratio']))

        print(
            best_row[
                [
                    'variant',
                    'predicted_p_ratio',
                    'mcmc_score',
                    'metaforge_p_ratio',
                    'bulk_modulus',
                    'shear_modulus',
                    'mechanically_valid',
                ]
            ].to_string()
        )

        display(valid.sort_values('metaforge_p_ratio').head(10))

In [ ]:
# Plot real MetaForge scan results for this one network.

if 'result_df' in globals() and isinstance(result_df, pd.DataFrame) and not result_df.empty:
    if 'support_scale_label' not in result_df.columns:
        result_df['support_scale_label'] = np.where(result_df['support_target'].eq('none'), 'none', 'scale ' + result_df['support_target'].astype(str) + ' ' + result_df['support_factor_resolved'].astype(str) + 'x')
    plot_df = result_df[result_df['variant'].ne('original')].copy()
    original_pr = float(result_df.loc[result_df['variant'].eq('original'), 'metaforge_p_ratio'].iloc[0])
    valid = plot_df[plot_df['mechanically_valid'].fillna(False)].copy()
    invalid = plot_df[~plot_df['mechanically_valid'].fillna(False)].copy()

    fig, axes = plt.subplots(1, 2, figsize=(11.2, 4.6), constrained_layout=True)
    ax = axes[0]
    for (soften_target, support_target), group in valid.groupby(['soften_target', 'support_target']):
        ax.scatter(group['soften_frac'], group['metaforge_p_ratio'], s=44, alpha=0.78, label=f'{soften_target} + {support_target}')
    ax.axhline(original_pr, color='#333333', lw=1.2, ls='--', label='original')
    ax.axhline(0, color='#777777', lw=1.0, ls=':')
    ax.set_xlabel('softened diagonal-edge fraction')
    ax.set_ylabel('MetaForge p-ratio')
    ax.set_title(f'sim {single_sim_idx}: valid candidates')
    ax.legend(frameon=False, fontsize=7, ncol=2)
    style_axes(ax)

    ax = axes[1]
    ax.scatter(valid['bulk_modulus'], valid['shear_modulus'], color='#4C78A8', alpha=0.75, label='valid')
    if not invalid.empty:
        ax.scatter(invalid['bulk_modulus'], invalid['shear_modulus'], color='#E45756', alpha=0.45, label='invalid')
    ax.axvline(cfg['valid_bulk_min'], color='#333333', lw=1, ls='--')
    ax.axhline(cfg['valid_shear_min'], color='#333333', lw=1, ls='--')
    ax.set_xlabel('bulk modulus')
    ax.set_ylabel('shear modulus')
    ax.set_title('Mechanical validity screen')
    ax.legend(frameon=False)
    style_axes(ax)

    fig.savefig(output_dir / f'single_network_{single_sim_idx:03d}_metaforge_scan.png', dpi=180)
    plt.show()

    best_table = valid.sort_values('metaforge_p_ratio').head(20)
    print(best_table[['variant', 'soften_target', 'soften_frac', 'support_scale_label', 'metaforge_p_ratio', 'bulk_modulus', 'shear_modulus']].round(6).to_string(index=False))
else:
    print('Run the MetaForge scan cell first.')


## Soft-Edge Geography in Low P-Ratio Reid Networks

This section does not modify networks. It measures where the naturally soft edges live in the original Reid data and compares low-p-ratio networks against high-p-ratio networks.

In [ ]:
# Research where soft / near-zero edges occur in low-p-ratio Reid networks.

soft_cfg = {
    'low_quantile': 0.20,
    'high_quantile': 0.80,
    'soft_abs_thresholds': [0.05, 0.075, 0.10, 0.15, 0.20],
    'soft_quantile_per_network': 0.20,
    'center_radius_cut': 0.35,
    'boundary_radius_cut': 0.75,
    'grid_bins': 5,
}

soft_output_dir = output_dir / 'soft_edge_geography'
soft_output_dir.mkdir(parents=True, exist_ok=True)


def box_xy_limits(graph):
    box = graph.box
    return float(box.x1), float(box.x2), float(box.y1), float(box.y2)


def edge_geography_rows(sim_idx, sim):
    graph = sim[0]
    xy = graph.x[:, :2].detach().cpu().numpy()
    x1, x2, y1, y2 = box_xy_limits(graph)
    cx = 0.5 * (x1 + x2)
    cy = 0.5 * (y1 + y2)
    sx = max(abs(x2 - x1), 1e-12)
    sy = max(abs(y2 - y1), 1e-12)
    edge_index = graph.edge_index.detach().cpu().numpy()
    edge_attr = graph.edge_attr.detach().cpu().numpy()
    seen = set()
    rows = []
    for e in range(edge_index.shape[1]):
        i, j = int(edge_index[0, e]), int(edge_index[1, e])
        key = tuple(sorted((i, j)))
        if key in seen:
            continue
        seen.add(key)
        pi = xy[i]
        pj = xy[j]
        midpoint = 0.5 * (pi + pj)
        vec = pj - pi
        angle = abs(np.arctan2(vec[1], vec[0]))
        angle = min(angle, np.pi - angle)
        if angle < np.deg2rad(25):
            orientation = 'horizontal'
        elif angle > np.deg2rad(65):
            orientation = 'vertical'
        else:
            orientation = 'diagonal'
        x_norm = (midpoint[0] - cx) / (0.5 * sx)
        y_norm = (midpoint[1] - cy) / (0.5 * sy)
        r_norm = float(np.sqrt(x_norm * x_norm + y_norm * y_norm))
        rows.append({
            'sim_idx': int(sim_idx),
            'edge_i': i,
            'edge_j': j,
            'stiffness': float(edge_attr[e, -1]),
            'mid_x': float(midpoint[0]),
            'mid_y': float(midpoint[1]),
            'x_norm': float(x_norm),
            'y_norm': float(y_norm),
            'r_norm': r_norm,
            'orientation': orientation,
            'angle_deg': float(np.rad2deg(angle)),
            'edge_length': float(np.linalg.norm(vec)),
            'region': 'center' if r_norm < soft_cfg['center_radius_cut'] else ('boundary' if r_norm > soft_cfg['boundary_radius_cut'] else 'middle'),
        })
    return rows


edge_rows = []
for sim_idx, sim in enumerate(data):
    edge_rows.extend(edge_geography_rows(sim_idx, sim))
edge_geo_df = pd.DataFrame(edge_rows)

if 'summary_df' not in globals() or summary_df.empty:
    summary_df = build_summary_from_data()
edge_geo_df = edge_geo_df.merge(summary_df[['sim_idx', 'final_p_ratio']], on='sim_idx', how='left')

low_cut = float(summary_df['final_p_ratio'].quantile(soft_cfg['low_quantile']))
high_cut = float(summary_df['final_p_ratio'].quantile(soft_cfg['high_quantile']))
edge_geo_df['p_ratio_group'] = np.select(
    [edge_geo_df['final_p_ratio'] <= low_cut, edge_geo_df['final_p_ratio'] >= high_cut],
    ['low_p_ratio', 'high_p_ratio'],
    default='middle_p_ratio',
)

network_q = edge_geo_df.groupby('sim_idx')['stiffness'].quantile(soft_cfg['soft_quantile_per_network']).rename('network_soft_q')
edge_geo_df = edge_geo_df.merge(network_q, on='sim_idx', how='left')
edge_geo_df['is_soft_network_q'] = edge_geo_df['stiffness'] <= edge_geo_df['network_soft_q']
for threshold in soft_cfg['soft_abs_thresholds']:
    col = f'is_soft_lt_{str(threshold).replace(".", "p")}'
    edge_geo_df[col] = edge_geo_df['stiffness'] < float(threshold)

edge_geo_df.to_csv(soft_output_dir / 'edge_geography_rows.csv', index=False)
print('low p-ratio cutoff:', low_cut)
print('high p-ratio cutoff:', high_cut)
print('wrote', soft_output_dir / 'edge_geography_rows.csv')


In [ ]:
# Summarize soft-edge distribution by p-ratio group.

soft_col = 'is_soft_lt_0p1'
summary_rows = []
for group_name, group in edge_geo_df.groupby('p_ratio_group'):
    row = {
        'p_ratio_group': group_name,
        'networks': int(group['sim_idx'].nunique()),
        'edges': int(len(group)),
        'mean_p_ratio': float(group.groupby('sim_idx')['final_p_ratio'].first().mean()),
        'soft_frac_lt_0p1': float(group[soft_col].mean()),
        'soft_frac_network_q20': float(group['is_soft_network_q'].mean()),
        'mean_stiffness': float(group['stiffness'].mean()),
        'stiffness_std_edges': float(group['stiffness'].std()),
        'soft_mean_r_norm': float(group.loc[group[soft_col], 'r_norm'].mean()),
        'all_mean_r_norm': float(group['r_norm'].mean()),
    }
    for orientation in ['horizontal', 'vertical', 'diagonal']:
        o = group[group['orientation'].eq(orientation)]
        row[f'{orientation}_edge_frac'] = float(len(o) / len(group)) if len(group) else np.nan
        row[f'{orientation}_soft_frac_lt_0p1'] = float(o[soft_col].mean()) if len(o) else np.nan
    for region in ['center', 'middle', 'boundary']:
        r = group[group['region'].eq(region)]
        row[f'{region}_edge_frac'] = float(len(r) / len(group)) if len(group) else np.nan
        row[f'{region}_soft_frac_lt_0p1'] = float(r[soft_col].mean()) if len(r) else np.nan
    summary_rows.append(row)
soft_group_summary_df = pd.DataFrame(summary_rows).sort_values('mean_p_ratio')
soft_group_summary_df.to_csv(soft_output_dir / 'soft_edge_group_summary.csv', index=False)
print(soft_group_summary_df.round(4).to_string(index=False))

orientation_summary_df = edge_geo_df.groupby(['p_ratio_group', 'orientation'], as_index=False).agg(
    networks=('sim_idx', 'nunique'),
    edges=('stiffness', 'size'),
    mean_p_ratio=('final_p_ratio', 'mean'),
    soft_frac_lt_0p1=(soft_col, 'mean'),
    mean_stiffness=('stiffness', 'mean'),
    mean_r_norm=('r_norm', 'mean'),
)
orientation_summary_df.to_csv(soft_output_dir / 'soft_edge_orientation_summary.csv', index=False)

region_summary_df = edge_geo_df.groupby(['p_ratio_group', 'region'], as_index=False).agg(
    networks=('sim_idx', 'nunique'),
    edges=('stiffness', 'size'),
    soft_frac_lt_0p1=(soft_col, 'mean'),
    mean_stiffness=('stiffness', 'mean'),
)
region_summary_df.to_csv(soft_output_dir / 'soft_edge_region_summary.csv', index=False)


In [ ]:
# Plot distributions: stiffness, orientation, radius, and spatial density of soft edges.

plot_groups = ['low_p_ratio', 'high_p_ratio']
fig, axes = plt.subplots(2, 2, figsize=(10.5, 8.0), constrained_layout=True)

ax = axes[0, 0]
for group_name, color in [('low_p_ratio', '#E45756'), ('high_p_ratio', '#4C78A8')]:
    vals = edge_geo_df.loc[edge_geo_df['p_ratio_group'].eq(group_name), 'stiffness'].to_numpy(float)
    ax.hist(vals, bins=40, density=True, alpha=0.45, color=color, label=group_name)
ax.axvline(0.1, color='#333333', ls='--', lw=1.2)
ax.set_xlabel('edge stiffness')
ax.set_ylabel('density')
ax.set_title('Edge stiffness distribution')
ax.legend(frameon=False)
style_axes(ax)

ax = axes[0, 1]
orient_plot = orientation_summary_df[orientation_summary_df['p_ratio_group'].isin(plot_groups)].copy()
orient_pivot = orient_plot.pivot(index='orientation', columns='p_ratio_group', values='soft_frac_lt_0p1').reindex(['horizontal', 'vertical', 'diagonal'])
orient_pivot.plot(kind='bar', ax=ax, color=['#4C78A8' if c == 'high_p_ratio' else '#E45756' for c in orient_pivot.columns])
ax.set_ylabel('fraction stiffness < 0.1')
ax.set_title('Soft-edge fraction by orientation')
ax.tick_params(axis='x', rotation=0)
ax.legend(frameon=False)
style_axes(ax)

ax = axes[1, 0]
for group_name, color in [('low_p_ratio', '#E45756'), ('high_p_ratio', '#4C78A8')]:
    vals = edge_geo_df.loc[edge_geo_df['p_ratio_group'].eq(group_name) & edge_geo_df[soft_col], 'r_norm'].to_numpy(float)
    ax.hist(vals, bins=np.linspace(0, 1.4, 25), density=True, alpha=0.45, color=color, label=group_name)
ax.set_xlabel('normalized edge-midpoint radius')
ax.set_ylabel('density among soft edges')
ax.set_title('Where soft edges sit radially')
ax.legend(frameon=False)
style_axes(ax)

ax = axes[1, 1]
region_plot = region_summary_df[region_summary_df['p_ratio_group'].isin(plot_groups)].copy()
region_pivot = region_plot.pivot(index='region', columns='p_ratio_group', values='soft_frac_lt_0p1').reindex(['center', 'middle', 'boundary'])
region_pivot.plot(kind='bar', ax=ax, color=['#4C78A8' if c == 'high_p_ratio' else '#E45756' for c in region_pivot.columns])
ax.set_ylabel('fraction stiffness < 0.1')
ax.set_title('Soft-edge fraction by radial region')
ax.tick_params(axis='x', rotation=0)
ax.legend(frameon=False)
style_axes(ax)

fig.savefig(soft_output_dir / 'soft_edge_distribution_summary.png', dpi=180)
plt.show()


In [ ]:
# Spatial heat maps of soft-edge midpoints for low vs high p-ratio networks.

bins = int(soft_cfg['grid_bins'])
fig, axes = plt.subplots(1, 3, figsize=(13.2, 4.1), constrained_layout=True)
heatmaps = []
for ax, group_name, title in zip(axes[:2], ['low_p_ratio', 'high_p_ratio'], ['Low p-ratio soft edges', 'High p-ratio soft edges']):
    group = edge_geo_df[edge_geo_df['p_ratio_group'].eq(group_name) & edge_geo_df[soft_col]].copy()
    heat, xedges, yedges = np.histogram2d(group['x_norm'], group['y_norm'], bins=bins, range=[[-1, 1], [-1, 1]])
    heat = heat.T
    heat = heat / max(float(heat.sum()), 1.0)
    heatmaps.append(heat)
    im = ax.imshow(heat, origin='lower', extent=[-1, 1, -1, 1], cmap='magma', aspect='equal')
    ax.set_title(title)
    ax.set_xlabel('normalized x')
    ax.set_ylabel('normalized y')
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

low_heat, high_heat = heatmaps
contrast = low_heat - high_heat
im = axes[2].imshow(contrast, origin='lower', extent=[-1, 1, -1, 1], cmap='coolwarm', aspect='equal')
axes[2].set_title('Low minus high soft-edge density')
axes[2].set_xlabel('normalized x')
axes[2].set_ylabel('normalized y')
fig.colorbar(im, ax=axes[2], fraction=0.046, pad=0.04)
fig.savefig(soft_output_dir / 'soft_edge_spatial_heatmaps.png', dpi=180)
plt.show()

spatial_rows = []
for group_name, heat in [('low_p_ratio', low_heat), ('high_p_ratio', high_heat), ('low_minus_high', contrast)]:
    for yi in range(heat.shape[0]):
        for xi in range(heat.shape[1]):
            spatial_rows.append({'group': group_name, 'grid_x': xi, 'grid_y': yi, 'value': float(heat[yi, xi])})
pd.DataFrame(spatial_rows).to_csv(soft_output_dir / 'soft_edge_spatial_grid.csv', index=False)


In [ ]:
# Edge-level signals: which locations/orientations correlate with p-ratio across networks?

network_rows = []
for sim_idx, group in edge_geo_df.groupby('sim_idx'):
    row = {'sim_idx': int(sim_idx), 'final_p_ratio': float(group['final_p_ratio'].iloc[0])}
    for threshold in soft_cfg['soft_abs_thresholds']:
        col = f'is_soft_lt_{str(threshold).replace(".", "p")}'
        row[f'soft_frac_lt_{str(threshold).replace(".", "p")}'] = float(group[col].mean())
    for orientation in ['horizontal', 'vertical', 'diagonal']:
        o = group[group['orientation'].eq(orientation)]
        row[f'{orientation}_mean_stiffness'] = float(o['stiffness'].mean()) if len(o) else np.nan
        row[f'{orientation}_stiffness_std'] = float(o['stiffness'].std()) if len(o) else np.nan
        row[f'{orientation}_soft_frac_lt_0p1'] = float(o[soft_col].mean()) if len(o) else np.nan
    for region in ['center', 'middle', 'boundary']:
        r = group[group['region'].eq(region)]
        row[f'{region}_mean_stiffness'] = float(r['stiffness'].mean()) if len(r) else np.nan
        row[f'{region}_soft_frac_lt_0p1'] = float(r[soft_col].mean()) if len(r) else np.nan
    # joint orientation-region descriptors
    for orientation in ['horizontal', 'vertical', 'diagonal']:
        for region in ['center', 'middle', 'boundary']:
            jr = group[group['orientation'].eq(orientation) & group['region'].eq(region)]
            row[f'{orientation}_{region}_soft_frac_lt_0p1'] = float(jr[soft_col].mean()) if len(jr) else np.nan
    network_rows.append(row)

soft_network_descriptor_df = pd.DataFrame(network_rows)
soft_network_descriptor_df.to_csv(soft_output_dir / 'soft_edge_network_descriptors.csv', index=False)

corr_rows = []
for col in soft_network_descriptor_df.columns:
    if col in ['sim_idx', 'final_p_ratio']:
        continue
    clean = soft_network_descriptor_df[[col, 'final_p_ratio']].replace([np.inf, -np.inf], np.nan).dropna()
    if len(clean) < 5 or clean[col].std() <= 1e-12:
        continue
    r = float(clean[col].corr(clean['final_p_ratio']))
    corr_rows.append({'descriptor': col, 'pearson_r': r, 'corr_r2': r * r, 'mean': float(clean[col].mean()), 'std': float(clean[col].std())})
soft_descriptor_corr_df = pd.DataFrame(corr_rows).sort_values('corr_r2', ascending=False)
soft_descriptor_corr_df.to_csv(soft_output_dir / 'soft_edge_descriptor_correlations.csv', index=False)
print(soft_descriptor_corr_df.head(20).round(4).to_string(index=False))


## Structure/Topology Differences

Compare negative and positive p-ratio Reid networks beyond stiffness: node count, edge count, degree, edge length, orientation mix, and geometry/stiffness coupling.

In [ ]:
# Geometry/topology descriptors for negative-vs-positive p-ratio networks.

structure_output_dir = output_dir / 'structure_topology_signal'
structure_output_dir.mkdir(parents=True, exist_ok=True)


def graph_structure_descriptors(sim_idx, sim):
    graph = sim[0]
    keys, stiffness = unique_edge_stiffness(graph)
    geom = unique_edge_geometry(graph, keys)
    xy = graph.x[:, :2].detach().cpu().numpy()
    deg = np.zeros(xy.shape[0], dtype=float)
    lengths = []
    for i, j in keys:
        deg[int(i)] += 1
        deg[int(j)] += 1
        lengths.append(float(np.linalg.norm(xy[int(j)] - xy[int(i)])))
    lengths = np.asarray(lengths, dtype=float)
    row = {
        'sim_idx': int(sim_idx),
        'final_p_ratio': float(summary_df.loc[summary_df['sim_idx'].eq(sim_idx), 'final_p_ratio'].iloc[0]),
        'nodes': int(xy.shape[0]),
        'edges_unique': int(len(keys)),
        'edge_density': float(2 * len(keys) / (xy.shape[0] * max(xy.shape[0] - 1, 1))),
        'degree_mean': float(deg.mean()),
        'degree_std': float(deg.std()),
        'degree_min': float(deg.min()),
        'degree_max': float(deg.max()),
        'edge_length_mean': float(lengths.mean()),
        'edge_length_std': float(lengths.std()),
        'edge_length_cv': float(lengths.std() / (lengths.mean() + 1e-12)),
        'x_span': float(xy[:, 0].max() - xy[:, 0].min()),
        'y_span': float(xy[:, 1].max() - xy[:, 1].min()),
        'xy_aspect': float((xy[:, 0].max() - xy[:, 0].min()) / (xy[:, 1].max() - xy[:, 1].min() + 1e-12)),
        'node_x_std': float(xy[:, 0].std()),
        'node_y_std': float(xy[:, 1].std()),
        'stiffness_mean': float(stiffness.mean()),
        'stiffness_std': float(stiffness.std()),
    }
    for orientation in ['horizontal', 'vertical', 'diagonal']:
        idx = geom.loc[geom['orientation'].eq(orientation), 'edge_idx'].to_numpy(dtype=int)
        row[f'{orientation}_edge_frac'] = float(len(idx) / len(keys)) if len(keys) else np.nan
        row[f'{orientation}_stiffness_mean'] = float(stiffness[idx].mean()) if len(idx) else np.nan
        row[f'{orientation}_stiffness_std'] = float(stiffness[idx].std()) if len(idx) else np.nan
        row[f'{orientation}_length_mean'] = float(lengths[idx].mean()) if len(idx) else np.nan
    for region in ['center', 'middle', 'boundary']:
        idx = geom.loc[geom['region'].eq(region), 'edge_idx'].to_numpy(dtype=int)
        row[f'{region}_edge_frac'] = float(len(idx) / len(keys)) if len(keys) else np.nan
        row[f'{region}_stiffness_mean'] = float(stiffness[idx].mean()) if len(idx) else np.nan
    # Couplings: where are stiffness and geometry aligned?
    row['stiffness_length_corr'] = float(pd.Series(stiffness).corr(pd.Series(lengths))) if len(stiffness) > 2 else np.nan
    row['stiffness_radius_corr'] = float(pd.Series(stiffness).corr(geom['r_norm'])) if len(stiffness) > 2 else np.nan
    row['stiffness_angle_corr'] = float(pd.Series(stiffness).corr(geom['angle_deg'])) if len(stiffness) > 2 else np.nan
    return row

structure_df = pd.DataFrame([graph_structure_descriptors(i, sim) for i, sim in enumerate(data)])
structure_df['p_ratio_group'] = np.where(structure_df['final_p_ratio'] < 0, 'negative', 'positive')
structure_df.to_csv(structure_output_dir / 'structure_topology_descriptors.csv', index=False)

corr_rows = []
for col in structure_df.columns:
    if col in ['sim_idx', 'final_p_ratio', 'p_ratio_group']:
        continue
    clean = structure_df[[col, 'final_p_ratio']].replace([np.inf, -np.inf], np.nan).dropna()
    if len(clean) < 5 or clean[col].std() <= 1e-12:
        continue
    r = float(clean[col].corr(clean['final_p_ratio']))
    corr_rows.append({'descriptor': col, 'pearson_r': r, 'corr_r2': r*r})
structure_corr_df = pd.DataFrame(corr_rows).sort_values('corr_r2', ascending=False)
structure_corr_df.to_csv(structure_output_dir / 'structure_topology_correlations.csv', index=False)

summary_by_group = structure_df.groupby('p_ratio_group').mean(numeric_only=True).reset_index()
summary_by_group.to_csv(structure_output_dir / 'structure_topology_group_means.csv', index=False)
print('negative networks:', int((structure_df['final_p_ratio'] < 0).sum()))
print('positive networks:', int((structure_df['final_p_ratio'] >= 0).sum()))
print('\nTop structure/topology correlations:')
print(structure_corr_df.head(20).round(4).to_string(index=False))
print('\nNegative vs positive group means for selected descriptors:')
cols = ['p_ratio_group', 'final_p_ratio', 'nodes', 'edges_unique', 'degree_mean', 'edge_length_mean', 'diagonal_edge_frac', 'diagonal_stiffness_std', 'stiffness_length_corr', 'stiffness_radius_corr']
print(summary_by_group[[c for c in cols if c in summary_by_group.columns]].round(4).to_string(index=False))


In [ ]:
# Plot selected structural differences.

if 'structure_df' in globals() and isinstance(structure_df, pd.DataFrame) and not structure_df.empty:
    metrics = ['edges_unique', 'degree_mean', 'edge_length_mean', 'diagonal_edge_frac', 'diagonal_stiffness_std', 'stiffness_length_corr']
    fig, axes = plt.subplots(2, 3, figsize=(12.0, 7.0), constrained_layout=True)
    for ax, metric in zip(axes.ravel(), metrics):
        data_to_plot = [
            structure_df.loc[structure_df['final_p_ratio'] < 0, metric].dropna().to_numpy(float),
            structure_df.loc[structure_df['final_p_ratio'] >= 0, metric].dropna().to_numpy(float),
        ]
        ax.boxplot(data_to_plot, labels=['negative', 'positive'], showfliers=False)
        ax.scatter(np.full(len(data_to_plot[0]), 1) + rng.normal(0, 0.035, len(data_to_plot[0])), data_to_plot[0], s=14, alpha=0.45, color='#E45756')
        ax.scatter(np.full(len(data_to_plot[1]), 2) + rng.normal(0, 0.035, len(data_to_plot[1])), data_to_plot[1], s=14, alpha=0.35, color='#4C78A8')
        ax.set_title(metric)
        style_axes(ax)
    fig.savefig(structure_output_dir / 'negative_vs_positive_structure_boxplots.png', dpi=180)
    plt.show()


## Notes

This notebook focuses on one Reid network. The current 10 candidates stay near the observed best diagonal-softening window (`0.18-0.28`) and add a few stiffness-rescaling variants. Uniform `all` scaling is included as a check, but p-ratio should mostly be invariant to pure global scaling; the more meaningful variants stiffen `non_diagonal` axial support edges while keeping diagonal soft hinges.
